In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
#
#  ┌─ 반드시 유지할 계약 ───────────────────────────────────────────────────────────┐
#  │ · answer_question(question: str) 함수 이름과 입력 형식                         │
#  │ · 반환값: {"answer": 문자열, "retrieved": [[문서명, 조번호], ...]}            │
#  │ · retrieved: 실제 답변에 사용한 근거를 관련도 순으로 1~4개                    │
#  │ · 전역 FastAPI app, GET /health, POST /answer                                 │
#  │ · Qwen2.5-Instruct 계열 생성 모델을 Colab T4에서 로컬 실행                    │
#  │ · 새 Colab T4 런타임에서 외부 준비 작업 없이 위에서 아래로 한 번 실행         │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  ┌─ 팀이 자유롭게 구현할 부분 ─────────────────────────────────────────────────────┐
#  │ · 1번 셀 안의 결과기 구현 방식과 필요한 패키지                                 │
#  │ · answer_question 함수 내부의 처리 방식                                        │
#  │   단, 위의 고정 계약과 아래의 금지 조건은 유지해야 합니다.                     │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  사용할 수 없는 방식
#    · Google Drive 마운트, 미리 업로드한 파일, 개인 컴퓨터 경로에 의존하는 코드
#    · 외부 생성형 LLM API, 원격 임베딩·리랭커, 원격 관리형 검색 서비스
#    · 실행 중 사람의 파일 업로드·문자 입력·버튼 클릭을 기다리는 코드
#    · torch 재설치, torch.compile
#    · 질문과 관계없이 약관 원문 전체를 매 질문의 프롬프트에 넣는 방식
#
#  주의
#    · 약관 원문을 확보하는 방법은 팀별 자유 구현입니다.
#    · 공개·비공개 답변 JSON은 2번 셀이 생성합니다. 1번 셀에서 직접 만들지 않습니다.
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
# retrieved에 기록하는 문서명은 아래 네 이름 중 하나를 그대로 사용합니다.
# 조번호는 3 또는 "제3조"처럼 채점기가 조번호를 식별할 수 있는 형태로 반환합니다.
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

# 정확한 모델 크기와 로딩 옵션은 자유지만 생성 모델은 이 계열을 사용합니다.
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen/Qwen2.5-7B-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — 함수 내부를 팀 코드로 교체합니다
# =====================================================================================
# 이 영역에는 팀이 필요한 패키지 설치와 결과기 구현 코드를 작성합니다.
# 구현 방법을 제한하지 않으며, 운영진은 아래 answer_question 함수만 호출합니다.

# 설치하기
!pip install -q rank_bm25 kiwipiepy sentence-transformers transformers accelerate bitsandbytes

# import
import re, json, threading
import torch, numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 전체 처리 흐름
# RAW 약관 → 조항 파싱/청킹 → BM25+Dense 하이브리드 검색 → focus 원문 선택
# → Qwen 답변 생성 → 누락 근거/법적 결론 보완 → 자연스러운 단일 문단 반환
# 조항과 검색 결과는 공통적으로 dict[str, object]를 사용하며 주요 키는 다음과 같습니다.
#   doc: 공식 문서명, article: 조번호, title: 조 제목, text: 조 본문, score: 검색 점수

RAW = {
    """카카오 통합 약관""":
"""
제1장 환영합니다!

제1조 목적

㈜카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 회사가 제공하는 다
양한 인터넷과 모바일 서비스에 더 가깝게 다가갈 수 있도록 카카오 서비스 및 Daum 서비스(이하 통칭하여
‘서비스’)에 통합 적용될 수 있는 카카오 통합 약관(이하 ‘본 약관’)을 마련하였습니다. 본 약관은 여러분이
서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으
므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다.
• '카카오 서비스'라 함은 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2)
카카오계정으로 이용하는 서비스(예: 브런치)를 의미하며, "Daum" 브랜드를 사용하는 서비스는 포함되
지 않습니다.
• ‘Daum 서비스’라 함은 회사가 제공하는 “Daum” 브랜드를 사용하는 서비스를 말합니다.

제2조 약관의 명시, 효력 및 변경
1. 본 약관의 내용은 회사가 제공하는 개별 서비스 또는 서비스 초기 화면에 게시하거나 기타의 방법으로 공
지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다.
2. 회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변
경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게
Daum 공지사항 또는 카카오 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게
여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정 또는 Daum 아이디
로 사용하는 이메일 주소로 이메일을 발송하거나, 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는
문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림 메시지를 띄우는 등 합리적으로 가능한 방법으로
변경사항을 공지 또는 통지하겠습니다.
3. 회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부
의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없
는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 제
14조 제1항에 따라 이용계약을 해지할 수 있습니다.

제3조 약관 외 준칙
본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 서비스의 개별 이용약관, 운영정책 및
규칙 등(이하 ‘세부지침’)의 규정에 따릅니다. 또한 본 약관과 세부지침의 내용이 충돌할 경우 세부지침에 따
릅니다.
제2장 카카오계정 및 Daum 아이디 연결 및 관리

제4조 카카오계정 또는 Daum 아이디 생성 및 연결

1. 카카오계정이란 여러분이 카카오 서비스를 사용하기 위하여 필요한 로그인 계정을 의미합니다. 카카오계정
은 여러분이 약관에 동의하고 카카오계정 생성을 위해 필요한 일정 정보를 입력하시면, 카카오가 입력된
일정 정보를 인증한 후 가입을 승낙하는 절차로 생성됩니다.
2. Daum 아이디란 여러분이 Daum 서비스에서 본인을 식별하기 위해 미리 등록한 문자, 특수문자, 숫자
등의 조합으로, 여러분이 Daum 서비스약관 또는 본 약관에 동의하고 회원등록에 필요한 필수사항을 입
력한 후 회원등록을 완료하면 회사가 승낙하는 절차로 생성됩니다. 다만, 여러분이 카카오계정으로 Dau
m 서비스를 이용하는 경우 자동으로 추천된 Daum 아이디가 생성될 수 있습니다.
3. 카카오 서비스를 이용하기 위하여 반드시 카카오계정이 필요한 것은 아니나, 어떤 카카오 서비스는 카카
오계정이 반드시 필요합니다. Daum 서비스약관에 동의한 Daum 아이디로는 Daum 서비스만 이용할
수 있습니다. 여러분이 Daum 서비스 중 카카오 서비스와 연결되는 기능을 이용하기 위해서는 카카오계
정으로 Daum서비스를 이용하거나, 카카오계정과 Daum 아이디와의 연결이 필요합니다. 카카오계정으로
Daum 서비스를 이용하거나, 카카오계정에 기존에 등록한 Daum 아이디를 연결하면 카카오 서비스와
Daum 서비스에 설정한 정보, 서비스 이용기록 등을 카카오 서비스와 Daum 서비스에서 모두 이용할
수 있습니다. 회사는 서비스 회원 정책의 변경 등의 사유가 발생하였을 때, 회원에게 안내 후 계정 연결
서비스를 종료할 수 있습니다.
4. 여러분이 카카오계정으로 Daum 서비스를 이용하거나, 카카오계정과 Daum 아이디를 연결하면서 본 약
관에 동의하면 그 이후부터는 본 약관만을 적용받게 되고, 카카오 서비스 약관 및 Daum 서비스 약관은
더 이상 적용되지 않습니다. 다만, 계정 연결 서비스를 종료할 경우에는 약관 변경에 따라 카카오 통합서
비스 약관 또는 카카오 서비스 약관 및 Daum 서비스 약관이 적용될 수 있습니다.
제5조 카카오계정 또는 Daum 아이디 생성 거절 및 유보
1. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디의 생성을 승낙하지 않을 수 있
습니다. 특히, 여러분이 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정
및/또는 Daum 아이디를 생성할 수 있습니다.
• 회사가 본 약관에 의해 여러분의 카카오계정 또는 Daum 아이디를 삭제하였던 경우
• 여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정 또는 Daum 아이디를
생성하려 한 경우
• 카카오계정 또는 Daum 아이디 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우
• 기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우
2. 만약, 여러분이 위 조건에 위반하여 카카오계정 및/또는 Daum 아이디를 생성한 것으로 판명된 때에는
회사는 즉시 여러분의 서비스 이용을 중단하거나 카카오계정 및 Daum 아이디를 삭제하는 등 적절한 제
한을 할 수 있습니다.
3. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디 생성을 유보할 수 있습니다.
• 제공 서비스 설비용량에 현실적인 여유가 없는 경우
• 서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우
• 기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우

제6조 카카오계정 또는 Daum 아이디 등의 관리
1. 카카오계정 및 Daum 아이디는 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정 및 D
aum 아이디를 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정 및 D
aum 아이디를 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여
러분의 카카오계정 및/또는 Daum 아이디를 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가
적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사
에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다.
2. 여러분은 서비스 내 설정 화면을 통하여 여러분의 정보를 열람하고 수정할 수 있습니다. 다만, 서비스의
제공 및 관리를 위해 필요한 카카오계정, Daum 아이디, 전화번호, 단말기 식별번호, 기타 본인확인정보
등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있
습니다. 여러분이 서비스 이용 신청 시 알려주신 내용에 변동이 있을 때, 직접 서비스에서 수정하거나 이
메일, 고객센터를 통하여 회사에 알려 주시기 바랍니다.
3. 여러분이 서비스 내 정보를 수정하지 않아 발생하는 손해에 대하여 회사는 책임을 부담하지 아니합니다.
제3장 서비스의 이용

제7조 다양한 서비스 제공 및 변경 등
1. 회사는 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등 여러분이 인터넷과 모바일로
즐길 수 있는 다양한 서비스를 제공합니다. 여러분은 스마트폰의 어플리케이션 스토어 등에서 서비스를
다운받아 설치하거나 직접 PC에 설치 혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데
회사는 여러분이 원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로 알려
드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도 개별적인 서비스 이용방법
을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내 및 고지사항에서 더 상세하게 안내하고 있
으니 언제든지 확인하여 주시기 바랍니다.
2. 회사는 여러분이 서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의 개인적이고 전 세계적이며
양도불가능하고 비독점적인 무상의 라이선스를 여러분에게 제공합니다. 단, 회사가 여러분에게 회사의 상
표 및 로고를 사용할 권리를 부여하는 것은 아니라는 점은 잊지 말아주시기 바랍니다.
3. 회사는 더 나은 서비스를 위하여 서비스에 필요한 소프트웨어의 업데이트 버전을 제공할 수 있습니다. 소
프트웨어의 업데이트에는 중요한 기능의 추가 또는 불필요한 기능의 제거 등이 포함되어 있습니다. 여러
분들도 서비스를 즐겁게 이용할 수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다.
4. 회사는 더 나은 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및
기타 광고를 비롯한 다양한 정보를 서비스에 표시하거나 여러분의 메일 계정으로 직접 발송할 수 있습니
다.
5. 서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 카카오 고객 센터로 알려주시기 바랍니
다.
6. 여러분이 서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신 이동통신사의 무선인
터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게 별도의 데이터 통신요금이 부과되는 점을
유의하여 주시기 바랍니다. 서비스 이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과
책임 하에 이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이 가입하신
이동통신사에 문의하시기 바랍니다.

제8조 서비스 이용 방법 및 주의점
1. 여러분은 서비스를 자유롭게 이용할 수 있으나, 아래와 같이 서비스를 잘못된 방법으로 이용할 수 없다는
점을 잊지 말아주셨으면 합니다.
• 여러분은 잘못된 방법으로 서비스의 제공을 방해하거나 회사가 안내하는 방법 이외의 다른 방법을 사용
하여 서비스에 접근할 수 없습니다.
• 다른 서비스 이용자의 정보를 무단으로 수집, 이용하거나 다른 사람들에게 제공하는 행위도, 수신자의
명시적 수신거부 의사에 반하여 또는 수신자의 명시적인 동의 없이 광고성 정보를 전송하거나 서비스를
영리 목적으로 이용하는 것도, 음란 정보나 저작권 침해, 회사나 제3자 등에 대한 허위의 사실을 게시하
는 정보 등 공서양속 및 법령에 위반되는 내용의 정보 등을 발송하거나 게시하는 행위도 금지됩니다.
• 회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담
보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시
도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위도 금지됩니다.
2. 여러분은 서비스의 이용권한, 기타 이용 계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할
수 없습니다.
3. 카카오는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우, 신고된 이용자의
음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다. 카카오는 이용자간 분쟁 조정, 민원
처리를 위한 목적에 한하여, 제 3 자는 법령에 따라 권한이 부여된 경우에 한하여 이 정보를 열람할 수
있습니다. 카카오는 부정이용 방지 및 관리의 목적에 따라 신고 접수시부터 3년간 해당 정보를 3년간 보
관 후 파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수 있습니다.
4. 혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행
위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시 삭제하거나 여러분의 서비스 이용을 잠시 또
는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오
작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하
게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니
다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할
수 있습니다.
5. 회사는 법령에서 정하는 기간 동안 여러분이 서비스를 이용하기 위해 로그인 혹은 접속한 기록이 없는경
우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기
타 유효한 수단으로 통지 후 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 서비스 이용
을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다.
6. 회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을 의미합니다)로부
터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안 조치를 합니다. 더불어 유관기관의 권
고가 있거나 이용자 보호를 위하여 필요하다고 판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능
을 제공합니다.
7. 본 조에서 정한 사항 및 그 밖에 서비스의 이용에 관한 자세한 사항은 서비스 운영정책 등 을 참고해 주시
기 바랍니다.

제9조 게시물의 관리
1. 여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하 ‘정보통신망법’)및 저작권법등
관련법에 위반되는 내용을 포함하는 경우, 권리자는 회사에 관련법이 정한 절차에 따라 해당 게시물의 게시
중단 및 삭제 등을 요청할 수 있으며, 회사는 관련법에 따라 조치를 취합니다.
2. 회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타 회사의 정책 및 관련
법에 위반되는 경우에는 관련법에 따라 해당 게시물에 대해 임시조치 등을 취할 수 있습니다.
3. 위와 관련된 세부절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가 정한 ‘권리침해 신고(카
카오 서비스, Daum 서비스)’절차에 따릅니다.

제10조 권리의 귀속 및 저작물의 이용
1. 여러분은 사진, 글, 정보, (동)영상, 카카오 서비스, Daum 서비스 또는 회사에 대한 의견이나 제안 등 콘
텐츠(이하 ‘게시물’)를 서비스에 게시할 수 있으며, 이러한 게시물에 대한 저작권을 포함한 지적재산권은
당연히 권리자가 계속하여 보유합니다.
2. 여러분은 카카오 서비스 또는 Daum 서비스에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신,
전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적이고 영구적인 라이선스를 회사에
게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는 서비스를 운영, 개선, 홍보하고
새로운 서비스를 개발하기 위한 범위 내에서 사용됩니다. 이러한 목적 범위 내에서 회사와 명시적인 업
무계약을 체결한 상대방 또는 다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 서비스의
개선 및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 본 라이선스는 여
러분이 서비스의 사용을 중단하거나 카카오계정 및/또는 Daum 아이디를 탈퇴한 후에도 존속하게 됩니
다. 일부 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을 제공할 수 있습니다(
다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있습니다). 또한
일부 서비스에서는 제공된 콘텐츠에 대한 회사의 사용 범위를 제한하는 설정이 있습니다.
3. 여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한 권리를 보유해야 합니다
. 이러한 권리를 보유하지 않아 발생하는 모든 문제에 대해서는 게시자가 책임을 부담하게 됩니다. 또한,
여러분은 음란하거나 폭력적이거나 기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없
습니다.
4. 회사는 여러분의 콘텐츠가 법령에 위반되거나 음란 또는 청소년에게 유해한 게시물, 차별 갈등을 조장하
는 게시물, 도배·광고·홍보·스팸성 게시물, 계정을 양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게
시물 등 이라고 판단되는 경우 이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를
검토할 의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해 게시중단요
청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및 이용제한 절차의 내용은 카카오 운
영정책에서 확인하실 수 있습니다.
5. 서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한 콘텐츠에 대해서는 콘텐
츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다. 여러분이 서비스를 이용하더라도 다른 이용
자의 콘텐츠에 대하여 어떠한 권리를 가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용
하기 위해서는 콘텐츠 소유자로부터 별도로 허락을 받아야 합니다.

제11조 유료 서비스의 이용
1. 회사는 무료로 서비스를 제공하고 있으나, 일부 서비스의 경우 유료로 제공할 수 있습니다. 예를 들면, 카
카오톡에서 친구들과 무료로 메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들
에게 보낼 수 있으며, Daum 메일은 무료로 이용할 수 있으나 프리미엄 메일은 유료로 이용할 수 있습니
다.
2. 여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후 이용하는 것을 원칙으로 합
니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제 방법은 핸드폰결제, 신용카드결제, 일반전화결
제, 계좌이체, 무통장입금, 선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있
을 수 있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당 서비스의 이용을
중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가 이루어집니다.
3. 회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할 수 있으며, 여러분
은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다.
4. 여러분 개인의 귀책사유로 이용요금을 환불하는 경우 일반적인 방법은 아래와 같습니다.
• 회사가 제공하는 유료서비스가 결제 후 1회의 이용만으로 서비스의 이용이나 구매가 완료되는 서비스인
경우 해당 서비스를 이용한 후에는 환불이 불가능합니다. 단, 1회의 구매 완료 후 그 사용기한이 무제한
인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내에만 환불이 가능하며 환불금액은
구입금액*(365-사용일수/365)로 합니다.
• 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준) 이하로 지속되는 서비스인 경우 해지일로부터
이용일수에 해당하는 금액을 제외한 나머지 금액을 환불합니다. 본 항의 규정은 일(1)개월 단위로 매월
결제되는 서비스의 경우에도 적용됩니다.
• 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준)을 초과하여 지속되는 서비스인 경우 해지일로
부터 이용일수에 해당하는 금액과 총 남은 이용일수의 10%를 제외한 금액을 환불합니다. 단, 유료 서
비스 이용 개시일로부터 7일 이내에 해지를 요구하는 경우 이용일수에 해당하는 금액만을 제외하고 환
불합니다.
5. 상기의 규정에도 불구하고 아래 각 호의 경우에는 여러분 개인이 결제한 전액을 환불합니다. 단, 1회의 구
매 완료 후 그 사용기한이 무제한인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내
일 경우에만 환불합니다.
• 여러분이 결제를 완료한 후 서비스를 이용한 내역이 없는 경우
• 서비스 장애 또는 회사가 제시한 최소한의 기술사양을 충족하였음에도 불구하고 회사의 귀책사유로 서
비스를 이용하지 못한 경우
• 여러분이 구매한 서비스가 제공되지 않은 경우
• 제공되는 서비스가 표시·광고 등과 상이하거나 현저한 차이가 있는 경우
• 제공되는 서비스의 결함으로 서비스의 정상적인 이용이 현저히 불가능한 경우
6. 여러분은 이용요금에 대하여 이의를 제기할 수 있습니다. 단, 이용요금에 관한 이의는 그 사유 발생을 안
날로부터 1월, 그 사유가 발생한 날로부터 3월 이내에 제기하여야 합니다.
7. 회사는 과오금이 발생한 경우 또는 전액 환불의 경우 이용대금의 결제와 동일한 방법으로 환불하여야 합
니다. 다만, 동일한 방법으로 환불이 불가능하거나 서비스의 중도해지로 인한 부분 환불 등의 경우에는 회
사가 정하는 별도의 방법으로 환불합니다. 회사는 환불 의무가 발생한 날로부터 3영업일 이내에 환불을
진행하며, 환불이 지연되는 경우 지연이자율은 연리 11%로 합니다. 단, 환불에 여러분의 협조가 필요한
경우에 여러분의 귀책사유로 인한 환불 지연에 대해서는 지연이자를 지급하지 않습니다. 환불에 소요되는
비용은 여러분의 귀책사유로 인한 환불의 경우에는 여러분이, 회사의 귀책사유로 인한 환불의 경우에는
회사가 각각 부담합니다.
8. 본 약관의 유료서비스 규정과 각 각 개별 유료서비스 약관의 내용이 충돌하는 경우 각 개별약관의 규정에
따릅니다.

제12조 게시판 이용 상거래
1. 여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우 전자상거래 등에서의 소비
자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을 준수하여야 합니다.
2. 여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련 분쟁이 발생하는 경우
, 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수 있는 장치를 마련합니다.
3. 회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의 신원정보를 확인하고, 여
러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에 따라 소비자피해 분쟁조정기구, 공정거래 위
원회, 시도지사 또는 시장 군수 구청장이 신원정보 제공을 요구하는 경우 이에 협조합니다.

제13조 서비스의 이용, 변경 및 종료
1. 회사는 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 장비의 유지·
보수를 위한 정기 또는 임시 점검 또는 다른 상당한 이유로 서비스의 제공이 일시 중단될 수 있으며, 이
때에는 미리 서비스 제공화면에 공지하겠습니다. 만약, 회사로서도 예측할 수 없는 이유로 서비스가 중단
된 때에는 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하고, 2시간 이
상 복구가 지연되는 경우 Daum 공지사항 또는 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에
게시하여 알려 드리겠습니다.
2. 회사의 서비스 제공을 위해 계약한 CP와의 계약 종료 및 변경, 서비스/회원 정책의 변경, 신규서비스 개
시 등의 사유로 서비스의 내용이 변경되거나, 서비스가 종료될 수도 있습니다. 서비스 변경 사항 또는 종
료는 개별 서비스의 화면 또는 공지사항 란에 게시하여 여러분들께 알려드리겠습니다. 여러분께 중대한
영향을 미치는 서비스 변경 사항이나 종료는 전자메일(전자메일이 없는 경우 서비스 내 알림 등 별도의
전자적 수단) 또는 전화번호로 문자메세지를 발송하는 방법 등으로 개별적으로 알려드리겠습니다.
이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이
필요할 수 있습니다.

제14조 이용계약 해지
1. 여러분이 서비스의 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 서비
스 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다.
2. 이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여
러분의 정보나 여러분이 작성한 게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제
3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물
을 추가하는 등의 경우에는 해당 게시물 및 댓글이 삭제되지 않으므로 반드시 해지신청 전에 삭제하신 후
탈퇴하시기 바랍니다.
3. 또한, 여러분은 다양한 서비스 중에서 일부 서비스만을 선택적으로 해지하실 수 있으며, 이 경우에는 해지
된 서비스에 대한 데이터만 삭제되며, 다른 서비스 이용을 위한 카카오계정 및 Daum 아이디는 삭제되지
않고 남아 있게 됩니다.
4. 유료서비스 이용계약의 해지는 여러분의 서비스 해지 신청 및 회사의 승낙에 의해 성립하게 되고, 환불할
금액이 있는 경우 환불도 이루어 지게 됩니다. 다만 각 개별 유료서비스에서 본 약관과 다른 계약해지 방
법 및 효과를 규정하고 있는 경우 각 개별약관의 규정에 따릅니다.
5. 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다. 다만,
일부 서비스의 경우 다시 이용계약을 체결함에 있어 시간적 제한 등이 따를 수 있으며 이에 대한 구체적
인 내용은 세부지침에서 확인하실 수 있습니다.

제15조 개인정보의 보호
여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서
비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이
별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하
셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항
은 Daum 개인정보처리방침과 카카오 개인정보처리방침을 참고하여 주십시오.

제16조 위치기반서비스 제공
1. 회사는 여러분의 실생활에 더욱 보탬이 되는 유용한 서비스를 제공하기 위하여 서비스에 위치기반서비스
를 포함시킬 수 있습니다.
2. 회사의 위치기반서비스는 여러분의 단말기기의 위치정보를 수집하는 위치정보사업자로부터 위치정보를 전
달받아 제공하는 무료서비스이며, 구체적으로는 아래와 같습니다.
• 여러분의 현재 위치 또는 특정 위치를 다른 이용자와 공유하거나 그와 관련된 게시물을 작성할 수 있도
록 하는 서비스(장소공유서비스)
• 여러분의 현재 위치를 이용한 생활 정보나 광고성 정보를 제공하는 서비스(정보제공서비스)
• 여러분이 보유하는 사진 등 콘텐츠에 기록되거나 콘텐츠와 결합된 위치정보를 활용하여 다른 이용자와
콘텐츠를 공유하도록 도와주는 서비스(콘텐츠공유서비스)
3. 여러분이 14세 미만 이용자로서 개인위치정보를 활용한 위치기반서비스를 이용하기 위해서는 회사는 여
러분의 개인위치정보를 이용 또는 제공하게 되며, 이 경우 부모님 등 법정대리인의 동의가 먼저 있어야
합니다. 만약 법정대리인의 동의 없이 위치기반서비스가 이용된 것으로 판명된 때에는 회사는 즉시 여러
분의 위치기반서비스 이용을 중단하는 등 적절한 제한을 할 수 있습니다.
4. 여러분(14세 미만 이용자의 법정대리인 포함)은 서비스와 관련된 개인위치정보의 이용, 제공 목적, 제공받
는 자의 범위 및 위치기반서비스의 일부에 대하여 동의를 유보하거나, 이용·제공에 대한 동의의 전부 또는
일부 철회할 수 있으며, 일시적인 중지를 요구할 수 있습니다. 회사는 위치정보의 보호 및 이용 등에 관
한 법률의 규정에 따라 개인위치정보 및 위치정보 이용·제공사실 확인자료를 6개월 이상 보관하며, 여러
분이 동의의 전부 또는 일부를 철회한 때에는 회사는 철회한 부분에 해당하는 개인위치정보 및 위치정보
이용·제공사실 확인자료를 지체 없이 파기하겠습니다.
5. 여러분(14세 미만 이용자의 법정대리인 포함)은 회사에 대하여 여러분에 대한 위치정보 이용·제공사실 확
인자료나, 여러분의 개인위치정보가 법령에 의하여 제3자에게 제공되었을 때에는 그 이유 및 내용의 열람
또는 고지를 요구할 수 있고, 오류가 있는 때에는 정정을 요구할 수 있습니다. 만약, 회사가 여러분의 개
인위치정보를 여러분이 지정하는 제3자에게 직접 제공하는 때에는 법령에 따라 개인위치정보를 수집한 스
마트폰 등으로 여러분에게 개인위치정보를 제공받는 자, 제공 일시 및 제공 목적을 즉시 통보하겠습니다.
6. 회사는 8세 이하의 아동 등(금치산자, 중증 정신장애인 포함)의 보호의무자가 개인위치정보의 이용 또는
제공에 서면으로 동의하는 경우에는 해당 본인의 동의가 있는 것으로 보며, 이 경우 보호의무자는 개인위
치정보주체의 권리를 모두 행사할 수 있습니다.
7. 만약 회사가 제공하는 위치기반서비스와 관련하여 여러분의 권리를 침해당했거나 권리행사가 필요한 경우
고객센터를 통해 도움을 받으실 수 있으며, 여러분과 회사 간의 위치정보와 관련한 분쟁에 대하여 협의가
어려운 때에는 여러분은 위치정보의 보호 및 이용 등에 관한 법률 제 28조 제2항 및 개인정보보호법 제
43조의 규정에 따라 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다.

제17조 인증서비스
1. 본 조에서 사용하는 용어의 정의는 다음과 같습니다.
• 인증서비스 : 회사가 제공하는 전자서명과 인증서를 활용한 일체의 서비스를 말합니다.
• 전자서명: 서명자의 신원을 확인하고 서명자가 해당 전자문서에 서명하였다는 사실을 나타내는데 이용하
기 위하여 전자문서에 첨부되거나 논리적으로 결합된 전자적 형태의 정보를 말합니다.
• 인증서: 인증서라 함은 회사가 인증서비스를 통하여 발급하는 전자서명생성정보가 회원에게 유일하게 속
한다는 사실 등을 확인하고 이를 증명하는 전자적 정보를 말합니다.
• 전자서명생성정보: 전자서명을 생성하기 위하여 이용하는 전자적 정보를 말합니다.
• 이용기관: 인증회원의 전자서명 및 인증서를 바탕으로 한 거래 등을 위하여 인증서비스를 이용하려는 제
3자를 말합니다.
• 인증회원 : 회사로부터 전자서명생성정보를 인증 받은 회원을 말합니다.
2. 회사는 전자서명생성정보 및 인증서를 발급하고, 전자서명과 인증서를 활용한 각종 서비스를 아래 각 호
와 같이 인증회원에게 제공합니다. 이 때 회사는 필요한 경우 인증서비스의 유형 및 종류를 추가하거나, 부가
서비스를 별도로 제공할 수 있습니다.
• 전자서명생성정보 및 인증서 발급
• 전자서명 및 인증서를 활용한 각종 서비스
• 이용기관 로그인 및 신원확인을 위한 간편인증
• 기타 전자서명인증업무 운영준칙에서 정하는 서비스
3. 회원은 회사가 정하는 방법에 따라 정확한 정보만을 제공하여 인증서비스에 가입하여야 하며, 인증서를
발급받음과 동시에 인증회원으로 전환됩니다. 인증서는 명의자 당 1개의 카카오계정에서 1대의 기기에만 발급
됩니다. 만일 인증회원이 다른 기기 또는 다른 카카오계정에서 인증서를 재발급하는 경우 기존에 발급받은 인
증서는 자동 폐지됩니다.
4. 인증회원은 회사가 정한 방법에 따라 인증서비스를 이용하여야 합니다. 또한 인증회원은 자신의 전자서명
생성정보와 인증서 및 이와 관련된 모든 정보를 안전하게 관리하고 인증서비스 이용 기간 중 회사에 제공한
정보 및 인증서에 포함된 정보가 정확하고 완전하게 유지되도록 하여야 합니다. 인증회원은 자신의 전자서명
생성정보와 인증서 및 이에 관련된 정보를 타인에게 양도, 증여, 판매, 사용 허락할 수 없으며, 분실, 훼손,
도난 또는 유출되거나 그러할 위험이 있다고 인지한 경우 즉시 그 사실을 회사에 통지하여야 합니다.
5. 회사는 다음 각 호의 경우 인증서의 신청 및 발급을 제한하거나 발급된 인증서를 인증회원의 동의 없이
폐지할 수 있습니다.
• 피성년후견인 또는 피한정후견인이 법정대리인의 동의 없이 가입한 경우
• 타인 명의의 신청 및 정보 도용 등 신청 내용이 허위의 사실이라 판단되는 경우
• 회사가 제시하는 인증 절차를 완료하지 못하거나, 회사가 정하지 않은 비정상적인 방법으로 시스템에 접
근하여 인증서비스에 가입하는 경우
• 회사로부터 이용 정지를 당하거나, 법령 또는 본 약관을 위반하는 등의 이유로 서비스 이용 계약이 해지
된 회원이 재이용신청을 하는 경우
• 기타 회원의 귀책사유로 발급이 곤란한 경우 또는 회사가 정한 이용신청 요건이 충족되지 않은 경우
6. 인증회원은 인증서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안 됩니다.
• 회사가 정하지 않은 비정상적인 방법으로 시스템에 접근하거나 인증서비스를 이용하는 행위
• 부정한 방법으로 인증서를 발급받거나 행사하는 등 인증서비스를 불법적 또는 부당한 용도로 사용하는
행위
7. 회사는 다음 각 호의 경우 인증회원에게 발급한 인증서 이용의 일부 또는 전부를 제한할 수 있으며, 인증
회원의 동의 없이 인증서를 즉시 폐지할 수 있습니다.
• 인증서의 유효기간이 경과한 경우
• 인증회원이 인증서의 비밀번호를 연속하여 제한 횟수 이상 잘못 입력한 경우
• 인증회원의 카카오계정에 등록된 카카오톡 전화번호가 변경된 경우
• 인증회원의 사망, 구속 등으로 신원확인이나 전자거래가 불가능한 경우
• 인증서비스 가입 시 본인확인기관에서 전달받은 연계정보(CI)가 국적, 성별 등의 변경으로 더이상 유효
하지 않음이 확인된 경우
• 회사가 인증서비스와 관련된 보안절차나 인증회원의 전자서명생성정보 유출과 같은 보안상의 이유로 기
발급된 인증서의 이용제한이 필요한 경우
• 전시, 사변, 천재지변 또는 이에 준하는 비상사태가 발생하거나 발생할 우려가 있는 경우
• 회사 고객센터 등을 통해서 인증서의 분실신고가 접수된 경우
• 인증회원의 인증서가 부정하게 사용된 사실을 회사가 인지한 경우 등 인증회원이 본 약관 및 운영정책
을 포함한 회사의 서비스 이용 정책을 위반하거나 위반할 우려가 있다고 회사가 판단하는 경우
• 기타 인증서비스의 안전성과 신뢰성을 저해할 우려가 있는 경우
8. 회사는 인증서를 사용하는 인증회원과 이용기관 상호간 거래에 대하여 어떠한 책임도 부담하지 않으며,
회사는 인증회원과 이용기관의 귀책사유로 인하여 발생한 손해에 대하여 회사의 귀책사유가 없는 경우 책임
을 부담하지 않습니다.
9. 본 조에서 정하고 있는 내용 외에 인증서비스와 관련된 상세한 사항은 전자서명법 등을 포함한 관련법령
및 회사가 별도로 정한 전자서명인증업무준칙에 따르며, 회사는 인증서비스 공지사항 및 고객센터 도움말 페
이지 등을 통하여 회원에게 안내합니다.
제4장 기타

제18조 손해배상 등
1. 회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항
에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이
작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지
않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다.
2. 회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 법령에 따라 여러분의 손해를 배
상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습
니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징
벌적 손해에 대한 책임을 부담하지 않습니다.
• 천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해
• 여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우
• 서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해
• 제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해
• 제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해
• 제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해
• 전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에
서 발생된 손해
• 기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해

제19조 청소년보호
모든 연령대가 자유롭게 이용할 수 있는 공간으로써 유해 정보로부터 청소년을 보호하고 청소년의 안전한 인
터넷 사용을 돕기 위해 정보통신망법에서 정한 청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은
서비스 초기 화면 등에서 확인할 수 있습니다.

제20조 통지 및 공지
회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진
할 수 있습니다. 회사는 카카오계정 또는 Daum 아이디로 사용하는 이메일 주소로 이메일을 발송하거나, 여
러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림
메시지를 띄우는 등 합리적으로 가능한 방법으로 여러분에게 공지 또는 통지하며, 서비스 이용자 전체에 대
한 공지는 칠(7)일 이상 서비스 공지사항 란에 게시함으로써 효력이 발생합니다.

제21조 분쟁의 해결
본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분
간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사
소송법의 관할법원에 소를 제기할 수 있습니다.
• 공고일자 : 2022년 8월 18일
• 시행일자 : 2022년 8월 25일
서비스(위치기반서비스 포함) 관련 문의사항이 있으시면 언제든지 고객센터에 방문 또는 연락해 주시기 바랍
니다.""",
    """카카오계정 약관""":
"""
제 1 장 환영합니다!
제 1 조 (목적)
주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 다양한 인터넷과 모바일 서비스를 좀 더 편리하게 이용할 수 있도록 회사 또는 관계사의 개별 서비스에 모두 접속 가능한 통합로그인계정 체계를 만들고 그에 적용되는 '카카오계정 약관(이하 '본 약관')을 마련하였습니다. 본 약관은 여러분이 카카오계정 서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다.

제 2 조 (약관의 효력 및 변경)
①본 약관의 내용은 카카오계정 웹사이트 또는 개별 서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다.
②회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.
③회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 이용계약을 해지할 수 있습니다.
제 3 조 (약관 외 준칙)
본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 개별 서비스의 이용약관, 운영정책 및 규칙 등(이하 ‘세부지침’)의 규정에 따릅니다.

제 4 조 (용어의 정의)
①본 약관에서 사용하는 용어의 정의는 다음과 같습니다.
1.카카오계정: 회사 또는 관계사가 제공하는 개별 서비스를 하나의 로그인계정과 비밀번호로 회원 인증, 회원정보 변경, 회원 가입 및 탈퇴 등을 관리할 수 있도록 회사가 정한 로그인계정 정책을 말합니다.
2.회원: 카카오계정이 적용된 개별 서비스 또는 카카오계정 웹사이트에서 본 약관에 동의하고, 카카오계정을 이용하는 자를 말합니다.
3.관계사: 회사와 제휴 관계를 맺고 카카오계정을 공동 제공하기로 합의한 법인을 말합니다. 개별 관계사는 카카오 기업사이트에서 확인할 수 있고 추후 추가/변동될 수 있으며 관계사가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다.
4.개별 서비스: 카카오계정을 이용하여 접속 가능한 회사 또는 관계사가 제공하는 서비스를 말합니다. 개별 서비스는 추후 추가/변동될 수 있으며 서비스가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다.
5.카카오계정 웹사이트: 회원이 온라인을 통해 카카오계정 정보를 조회 및 수정할 수 있는 인터넷 사이트를 말합니다.
6.카카오계정 정보 : 카카오계정을 이용하기 위해 회사가 정한 필수 내지 선택 입력 정보로서 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통해 정보 확인, 변경 처리 등을 관리할 수 있는 회원정보 항목을 말합니다.
7.이용기관 : 제11조에 따른 디지털카드서비스와 관련하여 디지털카드를 제출받거나 확인하여 자신의 업무, 영업에 활용하는 제3자를 말합니다.
제 2 장 카카오계정 이용계약
제 5 조 (계약의 성립)
①카카오계정 이용 신청은 개별 서비스 또는 카카오계정 웹사이트 회원가입 화면에서 여러분이 카카오계정 정보에 일정 정보를 입력하는 방식으로 이루어집니다.
②카카오계정 이용계약은 여러분이 본 약관의 내용에 동의한 후 본 조 제1항에서 정한 이용신청을 하면 회사가 입력된 일정 정보를 인증한 후 가입을 승낙함으로써 체결됩니다.
제 6 조 (카카오계정 이용의 제한)
①제5조에 따른 가입 신청자에게 회사는 원칙적으로 카카오계정의 이용을 승낙합니다. 다만, 회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지 않을 수 있습니다. 특히, 여러분이 만 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정을 생성할 수 있습니다.
1.회사가 본 약관 또는 세부지침에 의해 여러분의 카카오계정을 삭제하였던 경우
2.여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정을 생성하려 한 경우
3.카카오계정 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우
4.제공 서비스 설비 용량에 현실적인 여유가 없는 경우
5.서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우
6.기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우
7.회사로부터 회원자격정지 조치 등을 받은 회원이 그 조치기간에 이용계약을 임의로 해지하고 재이용을 신청하는 경우
8.기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우
②만약, 여러분이 위 조건에 위반하여 카카오계정을 생성한 것으로 판명된 때에는 회사는 즉시 여러분의 카카오계정 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한 제한을 할 수 있습니다.
제 3 장 카카오계정 이용
제 7 조 (카카오계정 제공)
①회사가 개별 서비스와 연동하여 카카오계정에서 제공하는 서비스(이하 “카카오계정 서비스” 또는 “서비스”) 내용은 아래와 같습니다.
1.통합로그인 : 카카오계정이 적용된 개별 서비스에서 하나의 카카오계정과 비밀번호로 로그인할 수 있는 통합 회원 인증 서비스를 이용할 수 있습니다.
2.SSO(Single Sign On): 웹브라우저나 특정 모바일 기기에서 카카오계정 1회 로그인으로 여러분이 이용 중인 개별 서비스간 추가 로그인 없이 자동 접속 서비스를 이용할 수 있습니다.
3.카카오계정 정보 통합 관리 : 개별 서비스 이용을 위해 카카오계정 정보를 통합 관리합니다. 또한, 여러분이 이용하고자 하는 개별 서비스의 유형에 따라 전문기관을 통한 실명확인 및 본인인증을 요청할 수 있고, 이를 카카오계정 정보로 저장합니다.
4.사업자/단체 카카오계정 : 사업자/단체 명의로 카카오 서비스를 이용하기 위해 만들어진 카카오계정으로서 해당 사업자/단체의 책임 하에 권한을 위임받은 담당자가 이용, 관리할 수 있는 계정 서비스입니다.
5.기타 회사가 제공하는 서비스
②회사는 더 나은 카카오계정 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 서비스화면 내에 표시하거나 여러분의 이메일로 전송할 수 있습니다. 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다.
제 8 조 (카카오계정 서비스의 변경 및 종료)
①회사는 카카오계정 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 아래 각 호의 경우 카카오계정 서비스의 전부 또는 일부를 제한하거나 중지할 수 있습니다.
1.카카오계정 서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우
2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 카카오계정 이용에 지장이 있는 경우
3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반 사정으로 카카오계정 서비스를 유지할 수 없는 경우
4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우
②전항에 의한 카카오계정 서비스 중단의 경우에는 미리 제14조에서 정한 방법으로 여러분에게 통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만, 회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지 서버 장애, 시스템 다운 등)로 서비스가 중단된 경우에는 사전 통지 내지 공지를 할 수 없습니다. 이러한 경우에도 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하되, 2시간 이상 복구가 지연될 시 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다.
제 9 조 (카카오계정 관리)
①카카오계정은 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정을 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정을 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여러분의 카카오계정을 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다.
②여러분은 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통하여 여러분의 카카오계정 정보를 열람하고 수정할 수 있습니다. 다만, 카카오계정 서비스의 제공 및 관리를 위해 필요한 카카오계정, 전화번호, 단말기 식별번호, 기타 본인확인정보 등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있습니다.
③여러분이 이용 신청 시 알려주신 내용에 변동이 있을 때, 전항에 따라 직접 수정하시거나, 고객센터를 통하여 회사에 알려 주시기 바랍니다. 여러분이 카카오계정 정보를 적시에 수정하지 않아 발생하는 문제에 대하여 회사의 고의 또는 과실이 없는 한 회사는 책임을 부담하지 아니합니다.
제 10 조 (사업자/단체 카카오계정)
①사업자/단체 카카오계정은 사업자등록번호 또는 고유번호가 있는 사업자/단체가 권한을 위임받은 담당자(이하 본조에서 ‘담당자’)를 통해 만들어 이용할 수 있습니다. 사업자/단체 카카오계정의 이용 및 관리에 관한 책임은 해당 사업자/단체에 있으며, 회사는 이와 관련한 책임을 지지 않습니다.
②사업자/단체 카카오계정은 계정 정보에 등록된 담당자 1인만 이용할 수 있으며, 이를 다른 사람에게 공유하는 것은 금지됩니다.
③사업자/단체 카카오계정은 개인 카카오계정으로 전환할 수 없고, 다른 개인 또는 법인 등 제3자에게 양도할 수 없습니다.
④사업자/단체 카카오계정은 일부 카카오 서비스의 가입 및 이용이 제한되며, 가입 및 이용이 제한되는 서비스는 정책에 따라 변경될 수 있습니다.
⑤사업자/단체 카카오계정의 정보 변경 또는 담당자 변경 요청에 대해 회사는 해당 계정에 대한 정당한 권한이 있는지 확인하기 위하여 일정한 증빙서류를 요청할 수 있습니다.
⑥사업자/단체 카카오계정은 사업자/단체에 귀속되는 것으로, 담당자는 해당 계정에 대해 권리를 주장할 수 없습니다.
⑦본 조에서 정하고 있는 내용 외에 사업자/단체 카카오계정과 관련된 상세한 사항은 사업자/단체 카카오계정 운영정책에 따르며, 회사는 게시판 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다.
제 11 조 (디지털카드 서비스)
①회사는 회사를 포함한 제휴 발급기관의 요청에 따라 회원의 카카오계정에 자격증명, 티켓, 아이템 등의 디지털카드를 발급하고 이를 제휴 이용기관에 제출하는 등의 활용을 할 수 있도록 하는 서비스(이하 ‘디지털카드 서비스’라 합니다)를 제공합니다.
②디지털카드는 회원 본인의 신청에 따라 발급되거나, 이용자가 일정한 조건을 충족한 경우 또는 발급기관의 요청을 받은 경우에는 자동으로 발급될 수 있습니다. 단, 회사는 디지털카드 발급 과정에서 추가적인 인증 또는 이용 등의 동의를 요청할 수 있고, 해외 거주 또는 외국인 회원의 경우 디지털카드의 발급 및 이용이 제한 될 수 있습니다.
③회사는 제휴 발급기관이 제공하는 정보를 디지털카드에 담거나 표시할 뿐, 제휴 발급기관이 발급한 디지털카드의 내용에 대한 검증 및 적법성 등에 대한 보증을 하지 않습니다.
④회사, 발급기관 또는 이용기관(이하 발급기관과 이용기관을 통칭하여 ‘제휴사’라 합니다)이 제공하는 서비스에서 회원의 디지털카드 정보를 조회하거나 표시, 노출할 수 있습니다.
⑤디지털카드는 발급기관의 필요와 요청에 따라 회수 또는 수정될 수 있습니다. 회수된 디지털카드 및 디지털카드에 담긴 정보는 복구할 수 없습니다.
⑥디지털카드에는 발급기관이 설정한 유효기간이 있으며, 유효기간이 경과하는 경우 제휴사의 정책에 따라 디지털카드의 기능을 사용할 수 없거나 기타 제출 등의 활용에 제한이 있을 수 있습니다. 또한, 디지털카드의 활용처는 제휴사의 사정에 따라 변동될 수 있고, 회사는 디지털카드의 영속성을 보장하지 않으며, 기능 외의 금전적 가치도 인정하지 않습니다.
⑦회원이 카카오계정을 탈퇴하는 경우 해당 카카오계정에 발급되어 있는 디지털카드는 삭제되고, 동일한 디지털카드의 재발급이 불가능할 수 있습니다.
⑧회원은 디지털카드 서비스를 이용함에 있어서 아래 각 호의 행위는 하여서는 안 됩니다.
1.서비스 이용 시 허위 사실을 기재하거나, 타인의 명의 및 정보를 도용하여 회사가 제공하는 서비스 또는 디지털카드를 이용하는 행위
2.디지털카드 정보를 회원 본인이 아닌 제3자가 사용하도록 대여하는 행위
3.유효하지 않은 디지털카드를 비정상적 목적으로 사용하는 행위
4.서비스에서 회사가 게시한 정보의 무단 변경 또는 회사가 정한 정보 이외의 정보(컴퓨터 프로그램 등)등의 송신 또는 게시하는 행위
5.회사가 정하지 않은 비정상적인 방법으로 서비스를 이용하거나 시스템에 접근하는 행위
6.회사가 정하지 않은 비정상적인 방법으로 부당하게 디지털카드를 주고 받는 행위(예: 디지털카드의 유상거래, 이용자간 합의되지 않은 전송에 따른 탈취 행위, 정상적으로 안내되지 않은 방법에 의한 거래 행위 등)
7.기타 관련법령, 회사의 약관 및 운영정책을 위반하여 회사나 제휴사 또는 다른 제3자에게 손해를 끼치거나 손해를 끼칠 것으로 합리적으로 예상되는 경우
⑨회사는 디지털카드의 활용과 관련하여 회원, 발급기관, 이용기관 간의 관계에서 어떠한 책임도 부담하지 않으며, 회사는 발급기관과 이용기관의 귀책사유로 인하여 회원에게 발생한 손해에 대하여 회사의 귀책사유가 없는 한 책임을 지지 않습니다.
⑩본 조에서 정하고 있는 내용 외에 디지털카드 서비스와 관련된 상세한 사항은 디지털카드 서비스 운영정책에 따르며, 회사는 서비스 공지사항 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다.
제 4 장 계약당사자의 의무
제 12 조 (회원의 의무)
①여러분이 카카오계정 서비스를 이용할 때 아래 각 호의 행위는 하여서는 안 됩니다.
1.이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 회원의 카카오계정 및 비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의 허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위
2.타인의 명예를 손상시키거나 불이익을 주는 행위
3.게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위
4.회사 또는 제3자의 저작권 등 기타 권리를 침해하는 행위
5.공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게 유포하는 행위
6.카카오계정 서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는 컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위
7.카카오계정 서비스의 운영을 고의로 방해하거나 안정적 운영을 방해할 수 있는 정보 및 수신자의 명시적인 수신거부의사에 반하여 광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위
8.회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위
9.타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위
10.다른 회원의 개인정보를 수집, 저장, 공개하는 행위
11.자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가할 목적으로 허위의 정보를 유통시키는 행위
12.윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위
13.수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는 영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는 행위
14.관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램 포함)의 전송 또는 게시 행위
15.회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의 명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위
16.컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할 목적으로 고안된 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을 포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위
17.기타 불법한 행위
②여러분은 서비스의 이용권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다.
③혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 여러분의 계정・서비스 이용을 잠시 또는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다.
④본 조에서 정한 사항 및 그 밖에 카카오계정 서비스의 이용에 관한 자세한 사항은 카카오 운영정책 등을 참고해 주시기 바랍니다.
제 13 조 (개인정보의 보호)
여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 카카오 개인정보처리방침을 참고하여 주십시오.

제 14 조 (회원에 대한 통지 및 공지)
회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의 경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.

제 5 장 이용계약 해지 등
제 15 조 (이용계약 해지)
①여러분이 카카오계정 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다.
②회사는 여러분이 카카오계정 서비스를 이용하기 위해 카카오계정 로그인 혹은 접속한 기록이 없는 경우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기타 유효한 수단으로 통지 후 여러분의 카카오계정 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 카카오계정 서비스 이용을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다. 이와 관련된 보다 자세한 사항은 카카오 운영정책의 서비스 장기 미이용 처리 정책을 참고하시기 바랍니다.
③이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여러분의 카카오계정 정보 및 카카오계정으로 이용하였던 개별 서비스 데이터는 삭제됩니다. 다만, 여러분이 개별 서비스 내에서 작성한 게시물 등 모든 데이터의 삭제와 관련한 사항은 개별 서비스의 약관에 따릅니다.
④이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다.
제 16 조 (손해배상)
①회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이 작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다.
②회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다.
1.천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해
2.여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우
3.서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해
4.제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해
5.제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해
6.제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해
7.전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해
8.기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해
제 17 조 (분쟁의 해결)
본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법상의 관할법원에 소를 제기할 수 있습니다.

공고일자 : 2026년 5월 13일
시행일자 : 2026년 5월 29일""",
    """카카오 위치정보 이용약관""":
"""제 1 조 (목적)
본 약관은 주식회사 카카오(이하 "회사")가 제공하는 사물위치정보 및 위치기반 서비스(이하, 위치정보 서비스)에 대해 회사와 서비스를 이용하는 이용자간의 권리·의무 및 책임사항, 기타 필요한 사항 규정을 목적으로 합니다.

제 2 조 (이용약관의 효력 및 변경)
①본 약관은 이용자가 본 약관에 동의하고 회사가 정한 절차에 따라 위치정보 서비스의 이용자로 등록됨으로써 효력이 발생합니다.
②이용자가 본 약관의 “동의하기” 버튼을 클릭하였을 경우 본 약관의 내용을 모두 읽고 이를 충분히 이해하였으며, 그 적용에 동의한 것으로 봅니다.
③회사는 위치정보 서비스의 변경사항을 반영하기 위한 목적 등으로 필요한 경우 관련 법령을 위배하지 않는 범위에서 본 약관을 수정할 수 있습니다.
④약관이 변경되는 경우 회사는 변경사항을 그 적용일자 최소 15일 전에 회사의 홈페이지 또는 서비스 공지사항 등(이하, 홈페이지 등)을 통해 공지합니다. 다만, 개정되는 내용이 이용자 권리의 중대한 변경을 발생시키는 경우 적용일 최소 30일 전에 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 발송하는 방법 등으로 개별적으로 고지합니다.
⑤회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 이용자의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 이용자가 개정약관에 동의하지 않을 경우 본 약관에 대한 동의를 철회할 수 있습니다.
제 3 조 (약관 외 준칙)
이 약관에 명시되지 않은 사항에 대해서는 위치 정보의 보호 및 이용 등에 관한 법률, 개인정보보호법, 전기통신사업법, 정보통신망 이용촉진 및 정보보호 등에 관한 법률 등 관계법령 및 회사가 정한 지침 등의 규정에 따릅니다.

제 4 조 (서비스의 내용)
회사는 위치정보사업자로부터 수집한 이용자의 위치정보 또는 직접 수집한 사물위치정보를 이용하여 아래와 같은 위치정보 서비스를 제공합니다.

①검색결과 제공 및 콘텐츠 추천 : 이용자의 위치나 경로를 바탕으로 관련 정보나 콘텐츠를 검색하거나 추천해주는 서비스를 제공합니다.
②생활편의 서비스 제공 : 이용자의 위치에 따른 길찾기, 경로 또는 이동수단 추천, 경로 안내 및 알림 서비스를 제공합니다.
③위치 기반 콘텐츠 분류(Geo Tagging) : 이용자가 작성한 게시글, 사진, 영상 등에 위치정보를 저장하거나, 위치를 기반으로 콘텐츠를 분류하는 기능을 제공합니다.
④위치기반 소셜 서비스 제공 : 내 위치를 다른 이용자와 공유하거나 콘텐츠 남기기 등 인터랙션을 포함한 위치 서비스를 제공합니다.
⑤위치기반 광고 : 이용자의 위치정보를 활용한 광고성 정보 안내, 검색 및 디스플레이 광고소재 제공, 맞춤형 광고를 제공합니다.
제 5 조 (서비스 이용요금)
회사가 제공하는 위치정보 서비스는 무료입니다.
단, 무선 서비스 이용 시 발생하는 데이터 통신료는 별도이며, 이용자가 가입한 각 이동통신사의 정책에 따릅니다.

제 6 조 (서비스의 변경・제한・중지)
①회사는 정책변경 또는 관련법령 변경 등과 같은 제반 사정을 이유로 위치기반서비스를 유지할 수 없는 경우 위치기반서비스의 전부 또는 일부를 변경·제한·중지할 수 있습니다.
②회사는 아래 각호의 경우에는 이용자의 서비스 이용을 제한하거나 중지시킬 수 있습니다.
1.이용자가 회사 서비스의 운영을 고의 또는 중과실로 방해하는 경우
2.서비스용 설비 점검, 보수 또는 공사로 인하여 부득이한 경우
3.전기통신사업법에 규정된 기간통신사업자가 전기통신 서비스를 중지했을 경우
4.국가비상사태, 서비스 설비의 장애 또는 서비스 이용의 폭주 등으로 서비스 이용에 지장이 있는 때
5.기타 중대한 사유로 인하여 회사가 서비스 제공을 지속하는 것이 부적당하다고 인정하는 경우
③회사가 제1항 및 제2항의 규정에 의하여 서비스 이용을 제한하거나 중지한 때에는 그 사유 및 제한기간 등을 회사 홈페이지 등을 통해 사전 공지하거나 이용자에게 통지합니다.
제 7 조 (개인위치정보의 이용 또는 제공)
①회사는 개인위치정보를 이용하여 위치기반서비스를 제공하는 경우 본 약관에 고지하고 동의를 받습니다.
②회사는 이용자의 동의 없이 개인위치정보를 제3자에게 제공하지 않으며, 제3자에게 제공하는 경우에는 제공받는 자 및 제공목적을 사전에 이용자에게 고지하고 동의를 받습니다.
③제2항에 따라 개인위치정보를 이용자가 지정하는 제3자에게 제공하는 경우 개인위치정보를 수집한 통신단말장치 또는 전자우편주소로 매회 이용자에게 제공받는 자, 제공일시 및 제공목적을 즉시 통지합니다. 단, 아래의 경우 이용자가 미리 특정하여 지정한 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통지합니다.
1.개인위치정보를 수집한 당해 통신단말장치가 문자, 음성 또는 영상의 수신기능을 갖추지 아니한 경우
2.이용자의 개인위치정보를 수집한 통신단말장치 외의 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통보할 것을 미리 요청한 경우
제 8 조 (위치정보 수집·이용·제공사실 확인자료의 보관)
회사는 위치정보의 보호 및 이용 등에 관한 법률 제16조 제2항에 근거하여 위치정보 수집·이용·제공사실 확인자료를 위치정보시스템에 자동으로 기록·보존하며, 해당 자료는 6개월간 보관합니다.

제 9 조 (개인위치정보의 보유 목적 및 보유기간)
회사는 위치기반서비스 제공을 위해 아래와 같이 개인위치정보를 보유합니다.

①본 약관 제4조 따른 위치기반서비스 이용 및 제공 목적 달성한 때에는 지체없이 개인위치정보를 파기합니다.
②다만, 이용자가 작성한 게시물 또는 콘텐츠와 함께 위치정보가 저장되는 서비스의 경우 해당 게시물 또는 콘텐츠의 보관기간 동안 개인위치정보가 보관됩니다.
③그 외 위치기반서비스 제공을 위해 필요한 경우 이용목적 달성을 위해 필요한 최소한의 기간 동안 개인위치정보를 보유할 수 있습니다.
④위 1, 2, 3항에도 불구하고 다른 법령 또는 위치정보법에 따라 보유해야하는 정당한 사유가 있는 경우에는 그에 따릅니다.
제 10 조 (개인위치정보주체의 권리)
①이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 유보할 수 있습니다.
②이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 철회할 수 있습니다. 이 경우 회사는 지체 없이 철회된 범위의 개인위치정보 및 위치정보 이용·제공사실 확인자료를 파기합니다.
③이용자는 개인위치정보의 이용·제공의 일시적인 중지를 요구할 수 있습니다. 이 경우 회사는 이를 거절할 수 없으며 이를 충족하는 기술적 수단을 마련합니다
④이용자는 회사에 대하여 아래 자료에 대한 열람 또는 고지를 요구할 수 있으며, 해당 자료에 오류가 있는 경우에는 정정을 요구할 수 있습니다. 이 경우 회사는 정당한 사유 없이 요구를 거절하지 않습니다.
1.이용자에 대한 위치정보 이용·제공사실 확인자료
2.이용자의 개인위치정보가 위치정보의 보호 및 이용 등에 관한 법률 또는 다른 법령의 규정에 의하여 제3자에게 제공된 이유 및 내용
⑤이용자는 권리행사를 위해 본 약관 제14조의 연락처를 이용하여 회사에 요청할 수 있습니다.
제 11 조 (법정대리인의 권리)
①회사는 14세 미만의 이용자에 대해서는 개인위치정보를 이용한 위치기반서비스 제공 및 개인위치정보의 제3자 제공에 대한 동의를 이용자 및 이용자의 법정대리인으로부터 받아야 합니다. 이 경우 법정대리인은 본 약관 제10조에 의한 이용자의 권리를 모두 가집니다.
②회사는 14세 미만의 아동의 개인위치정보 또는 위치정보 이용, 제공사실 확인자료를 이용약관에 명시 또는 고지한 범위를 넘어 이용하거나 제3자에게 제공하고자 하는 경우 이용자와 이용자의 법정대리인의 동의를 받아야 합니다. 단, 아래의 경우는 제외합니다.
1.위치정보 및 위치기반서비스 제공에 따른 요금정산을 위하여 위치정보 이용, 제공사실 확인자료가 필요한 경우
2.통계작성, 학술연구 또는 시장조사를 위하여 특정 개인을 알아볼 수 없는 형태로 가공하여 제공하는 경우
제 12 조 (8세 이하의 아동 등의 보호의무자의 권리)
①회사는 아래의 경우에 해당하는 자(이하 “8세 이하의 아동 등”)의 위치정보의 보호 및 이용 등에 관한 법률 제26조2항에 해당하는 자(이하 “보호의무자”)가 8세 이하의 아동 등의 생명 또는 신체보호를 위하여 개인위치정보의 이용 또는 제공에 동의하는 경우에는 본인의 동의가 있는 것으로 봅니다.
1.8세 이하의 아동
2.피성년후견인
3.장애인복지법 제2조제2항제2호에 따른 정신적 장애를 가진 사람으로서 장애인고용촉진 및 직업재활법 제2조제2호에 따른 중증장애인에 해당하는 사람(장애인복지법 제32조에 따라 장애인 등록을 한 사람만 해당한다)
②8세 이하의 아동 등의 생명 또는 신체의 보호를 위하여 개인위치정보의 이용 또는 제공에 동의를 하고자 하는 보호의무자는 서면동의서에 보호의무자임을 증명하는 서면을 첨부하여 회사에 제출하여야 합니다.
③보호의무자는 8세 이하의 아동 등의 개인위치정보 이용 또는 제공에 동의하는 경우 본 약관 제9조에 의한 이용자의 권리를 모두 가집니다.
제 13 조 (손해배상)
회사의 위치정보의 보호 및 이용 등에 관한 법률 제15조 및 26조의 규정을 위반한 행위로 인해 손해를 입은 경우 이용자는 회사에 손해배상을 청구할 수 있습니다. 회사는 고의, 과실이 없음을 입증하지 못하는 경우 책임을 면할 수 없습니다.

제 14 조 (면책)
①회사는 다음 각 호의 경우로 위치기반서비스를 제공할 수 없는 경우 이로 인하여 이용자에게 발생한 손해에 대해서는 회사의 고의 과실이 없는 한 책임을 부담하지 않습니다.
1.천재지변 또는 이에 준하는 불가항력의 상태가 있는 경우
2.위치기반서비스 제공을 위하여 회사와 서비스 제휴계약을 체결한 제3자의 고의적인 서비스 방해가 있는 경우
3.이용자의 귀책사유로 위치기반서비스 이용에 장애가 있는 경우
4.제1호 내지 제3호를 제외한 기타 회사의 고의·과실이 없는 사유로 인한 경우
②회사는 위치기반서비스 및 위치기반서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며 이로 인해 발생한 이용자의 손해에 대하여는 회사의 고의 과실이 없는 한 책임을 부담하지 아니합니다.
제 15 조 (분쟁의 조정 및 기타)
①회사는 위치정보와 관련된 분쟁의 해결을 위해 이용자와 성실히 협의합니다.
②전항의 협의에서 분쟁이 해결되지 않은 경우, 회사와 이용자는 위치정보의 보호 및 이용 등에 관한 법률 제28조의 규정에 의해 방송통신위원회에 재정을 신청하거나, 개인정보보호법 제43조의 규정에 의해 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다.
제 16 조 (사업자 및 위치정보관리책임자 정보)
① 회사의 상호, 주소 및 연락처는 아래와 같습니다.

상호 : 주식회사 카카오
주소 : 제주특별자치도 제주시 첨단로 242 (영평동)
대표전화 : 1577-3754 (유료)
② 회사는 개인위치정보를 적절히 관리·보호하고, 이용자의 불만을 원활히 처리할 수 있도록 실질적인 책임을 질 수 있는 지위에 있는 자를 위치정보관리책임자로 지정해 운영하고 있습니다. 위치정보관리책임자는 위치기반서비스를 제공하거나 관리하는 부서의 부서장으로서 성명과 연락처는 아래와 같습니다.

성명 : 김연지
대표전화 : 1577-3754 (유료)
<시행일자>
공고일자 : 2026년 7월 2일
시행일자 : 2026년 7월 16일

""",
    """카카오 통합서비스약관""":
"""
제 1 장 환영합니다!
제 1 조 (목적 및 정의)
주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이
회사가 제공하는 다양한 인터넷과 모바일 서비스(이하 해당 서비스들을 모두 합하여 “통합서비스”
또는 “서비스”라 함)에 더 가깝고 편리하게 다가갈 수 있도록 ‘카카오 통합서비스약관’(이하 ‘본
약관’)을 마련하였습니다. 여러분은 본 약관에 동의함으로써 통합서비스에 가입하여 통합서비스를
이용할 수 있습니다. 단, 여러분은 회사가 아닌 계열사를 포함한 제 3 자가 제공하는 서비스 (예:
㈜카카오모빌리티가 제공하는 카카오 T 택시 서비스)에 가입되지는 않으며, 회사가 제공하는
유료서비스의 경우 여러분이 별도의 유료이용약관에 대한 동의한 때에 회사와 여러분 간의
유료서비스 이용계약이 성립합니다. 본 약관은 여러분이 통합서비스를 이용하는 데 필요한 권리,
의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서
주의 깊게 읽어주시기 바랍니다.
• 통합서비스: 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2)
카카오계정으로 이용하는 서비스(예: 브런치) (단, 서비스 명칭에 ‘카카오’가 사용되더라도
회사가 아닌 카카오 계열사에서 제공하는 서비스 (예: 카카오 T 택시 서비스)는 본 약관의
통합서비스에 포함되지 않습니다)
• 개별 서비스: 통합서비스를 구성하는 세부 하위 서비스를 의미하며, 예를 들어 각
통합서비스 내의 유료서비스, 카카오톡 서비스 등을 의미함
제 2 조 (약관의 효력 및 변경)
① 본 약관의 내용은 통합서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본
약관에 동의한 여러분 모두에게 그 효력이 발생합니다.
② 회사는 필요한 경우 관련 법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수
있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15 일 전부터
여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게
여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30 일 전부터 카카오계정에
등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스
내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한
휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려
드리겠습니다.
③ 회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관
시행일 7 일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게
고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로
봅니다.
④ 여러분은 변경된 약관에 대하여 거부의사를 표시함으로써 이용계약의 해지를 선택할 수
있습니다.
⑤ 본 약관은 여러분이 본 약관에 동의한 날로부터 본 약관 제 13 조에 따른 이용계약의
해지 시까지 적용하는 것을 원칙으로 합니다. 단, 본 약관의 일부 조항은 이용계약의
해지 후에도 유효하게 적용될 수 있습니다.
제 3 조 (약관 외 준칙)
본 약관에 규정되지 않은 사항에 대해서는 관련 법령 또는 통합서비스를 구성하는 개별 서비스의
이용약관, 운영정책 및 규칙, 카카오 운영정책 및 규칙 등(이하 총칭하여 ‘세부지침’)의 규정에
따릅니다. 세부지침은 본 약관과 더불어 이용계약의 일부를 구성합니다.
제 2 장 통합서비스 이용계약
제 4 조 (계약의 성립)
① 통합서비스에 가입하기 위해서는 카카오계정이 필요합니다. 카카오계정이 없으신 경우
카카오계정을 먼저 생성하시기 바랍니다.
② 통합서비스 이용계약은 여러분이 본 약관의 내용에 동의한 후 회사가 여러분의
카카오계정 정보 등을 확인한 후 승낙함으로써 체결됩니다.
제 5 조 (통합서비스 가입의 제한)
① 제 4 조에 따른 가입 신청자에게 회사는 원칙적으로 통합서비스 가입을 승낙합니다. 다만,
회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지
않을 수 있습니다. 특히, 여러분이 만 14 세 미만인 경우에는 부모님 등 법정대리인의
동의가 있는 경우에만 통합서비스에 가입할 수 있습니다.
1. 여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 통합서비스에
가입하려고 한 경우
2. 통합서비스 제공 설비 용량에 현실적인 여유가 없는 경우
3. 통합서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우
4. 기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우
5. 회사로부터 통합서비스 이용정지 조치 등을 받은 자가 그 조치기간에 통합서비스
이용계약을 임의로 해지하고 재가입을 신청하는 경우
6. 기타 관련 법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우
② 만약, 여러분이 위 조건에 위반하여 통합서비스에 가입한 것으로 판명된 때에는 회사는
즉시 여러분의 통합서비스 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한
제한을 할 수 있습니다.
제 3 장 통합서비스 이용
제 6 조 (다양한 서비스의 제공)
① 통합서비스 이용계약이 성립되면, 여러분은 통합서비스를 구성하는 개별 서비스를
여러분이 원하는 때에 자유롭게 이용할 수 있습니다.
② 다만, 통합서비스 내에서도 일부 개별 서비스의 경우 별도의 이용약관에 동의해야
이용이 가능하며 필요한 추가 정보를 기재하거나, 이메일 주소 승인 또는 문자메시지
인증, 인증서 발급 등 회사가 정한 인증 절차를 완료하여야 서비스 이용이 가능합니다.
③ 여러분은 통합서비스 가입 후에도 언제든지 통합서비스를 구성하는 개별 서비스 화면
또는 메뉴에서 제공하는 기능을 이용하여 해당 개별 서비스의 이용을 종료할 수 있으며,
이 경우 관련 법령에서 정하는 바에 따라 일정기간 보관해야 하는 정보를 제외하고는
해당 서비스 이용기록, 여러분이 작성한 게시물 등 모든 데이터는 즉시 삭제 처리됩니다.
다만, 여러분이 작성한 게시물이 제 3 자에 의하여 스크랩 또는 다른 공유 기능으로
게시되거나, 여러분이 제 3 자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는
해당 게시물 및 댓글은 삭제되지 않으므로 반드시 이용 종료 전에 삭제하시기 바라며,
일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도
있으니 이 점 유의하여 주시기 바랍니다. 개별 서비스 이용 종료 시점에 향후 일정기간
해당 서비스의 재이용에 제한이 있을 수 있다는 별도 안내가 있는 경우 해당 안내에
따라 해당 서비스의 재이용에 일정한 시간적 제한이 있을 수 있는 점 또한 유의하여
주시기 바랍니다.
④ 전항에 따른 개별 서비스의 이용 종료는 해당 개별 서비스의 이용 종료만을 의미하며,
통합서비스를 구성하는 다른 서비스의 이용이 종료되지는 않습니다. 여러분이
통합서비스 전체의 이용을 종료하고 싶은 경우에는 본 약관 제 13 조에서 정한 바처럼
통합서비스 이용계약을 해지하여야 합니다.
⑤ 회사는 여러분에게 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등
여러분이 인터넷과 모바일로 즐길 수 있는 다양한 서비스를 제공합니다. 여러분은
스마트폰의 어플리케이션 스토어 등에서 서비스를 다운받아 설치하거나 직접 PC에 설치
혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데 회사는 여러분이
원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로
알려드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도
개별적인 서비스 이용방법을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내
및 고지사항에서 더 상세하게 안내하고 있으니 언제든지 확인하여 주시기 바랍니다.
⑥ 회사는 여러분이 통합서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의
개인적이고 전 세계적이며 양도불가능하고 비독점적인 무상의 라이선스를 여러분에게
제공합니다. 단, 회사가 여러분에게 회사의 상표 및 로고를 사용할 권리를 부여하는
것은 아니라는 점은 잊지 말아주시기 바랍니다.
⑦ 회사가 여러분에게 제공하는 통합서비스에는 인공지능에 기반하여 운용되는 서비스가
포함될 수 있으며, 회사가 인공지능에 의하여 생성된 결과물을 제공할 경우에는 관련
법에 따라 고지 및 표시합니다.
⑧ 회사는 더 나은 통합서비스를 위하여 통합서비스에 필요한 소프트웨어의 업데이트
버전을 제공할 수 있습니다. 소프트웨어의 업데이트에는 중요한 기능의 추가 또는
불필요한 기능의 제거 등이 포함되어 있습니다. 여러분들도 통합서비스를 즐겁게 이용할
수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다.
⑨ 회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을
의미합니다)로부터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안
조치를 합니다. 더불어 유관기관의 권고가 있거나 이용자 보호를 위하여 필요하다고
판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능을 제공합니다.
⑩ 회사는 더 나은 통합서비스의 제공을 위하여 여러분에게 통합서비스의 이용과 관련된
각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 통합서비스 내에
표시하거나 여러분의 카카오계정 정보에 등록되어 있는 연락처로 직접 발송할 수
있습니다. 단, 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다.
⑪ 통합서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 고객센터로
알려주시기 바랍니다.
⑫ 여러분이 통합서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신
이동통신사의 무선인터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게
별도의 데이터 통신요금이 부과될 수 있는 점을 유의하여 주시기 바랍니다. 통합서비스
이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과 책임 하에
이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이
가입하신 이동통신사에 문의하시기 바랍니다.
제 7 조 (통합서비스의 변경 및 종료)
① 회사는 통합서비스를 365 일, 24 시간 쉬지 않고 제공하기 위하여 최선의 노력을
다합니다. 다만, 아래 각 호의 경우 통합서비스의 전부 또는 일부를 제한하거나 중지할
수 있습니다.
1.통합서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우
2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 통합서비스 이용에
지장이 있는 경우
3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반
사정으로 통합서비스의 전부 또는 일부를 유지할 수 없는 경우
4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우
② 전항에 의한 통합서비스 중단의 경우에는 미리 제 17 조에서 정한 방법으로 여러분에게
통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스
이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만,
회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지
서버 장애, 시스템 다운 등)로 인해 사전 통지 내지 공지가 불가능한 경우에는
그러하지 아니합니다. 이러한 경우에도 회사는 상황을 파악하는 즉시 최대한 빠른
시일 내에 서비스를 복구하도록 노력하고, 2 시간 이상 복구가 지연되는 경우 서비스
공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다.
제 8 조 (게시물의 관리)
① 여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하
‘정보통신망법’)및 저작권법 등 관련 법령에 위반되는 내용을 포함하는 경우, 권리자는
회사에 관련 법령이 정한 절차에 따라 해당 게시물의 게시중단 및 삭제 등을 요청할
수 있으며, 회사는 관련 법령에 따라 조치를 취합니다.
② 회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타
회사의 정책 및 관련 법령에 위반되는 경우에는 관련 법령에 따라 해당 게시물에 대해
임시조치 등을 취할 수 있습니다.
③ 위와 관련된 세부 절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가
정한 권리침해 신고 절차 에 따릅니다.
제 9 조 (권리의 귀속 및 저작물의 이용)
① 여러분은 사진, 글, 정보, (동)영상, 통합서비스 또는 회사에 대한 의견이나 제안 등
콘텐츠(이하 ‘게시물’)를 통합서비스 내에 게시할 수 있으며, 이러한 게시물에 대한
저작권을 포함한 지적재산권은 당연히 권리자가 계속하여 보유합니다.
② 여러분은 통합서비스 내에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신,
전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적인 라이선스를
회사에게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는
통합서비스를 운영, 개선, 홍보하고 새로운 서비스를 개발하기 위한 범위 내에서
사용되며, 이러한 목적 범위 내에서 회사와 명시적인 업무계약을 체결한 상대방 또는
다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 통합서비스의 개선
및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 일부
개별 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을
제공할 수 있습니다(다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의
삭제가 불가능할 수도 있습니다). 또한 일부 서비스에서는 제공된 콘텐츠에 대한
회사의 사용 범위를 제한하는 설정이 있습니다.
③ 여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한
권리를 보유해야 합니다. 이러한 권리를 보유하지 않아 발생하는 모든 문제에
대해서는 게시자가 책임을 부담하게 됩니다. 또한, 여러분은 음란하거나 폭력적이거나
기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없습니다.
④ 회사는 여러분의 콘텐츠가 관련 법령에 위반되거나 음란 또는 청소년에게 유해한
게시물, 차별 갈등을 조장하는 게시물, 도배 · 광고 · 홍보 · 스팸성 게시물, 계정을
양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게시물 등이라고 판단되는 경우
이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를 검토할
의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해
게시중단 요청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및
이용제한 절차의 내용은 카카오 운영정책에서 확인하실 수 있습니다.
⑤ 통합서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한
콘텐츠에 대해서는 콘텐츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다.
여러분이 통합서비스를 이용하더라도 다른 이용자의 콘텐츠에 대하여 어떠한 권리를
가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용하기 위해서는
콘텐츠 소유자로부터 별도로 허락을 받아야 합니다.
제 10 조 (유료 서비스의 이용)
① 통합서비스를 구성하는 개별 서비스의 대부분은 무료로 제공하고 있으나, 일부 개별
서비스는 유료로 제공될 수 있습니다. 예를 들면, 카카오톡에서 친구들과 무료로
메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들에게 보낼
수 있습니다.
② 여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후
이용하는 것을 원칙으로 합니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제
방법은 핸드폰결제, 신용카드결제, 일반전화결제, 계좌이체, 무통장입금,
선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있을 수
있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당
서비스의 이용을 중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가
이루어집니다.
③ 회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할
수 있으며, 여러분은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다.
④ 본 조에서 정하지 않은 내용은 개별 서비스에 적용되는 유료서비스 약관(예: 카카오
유료서비스 이용약관 등)에서 정하며, 본 조의 내용과 개별 서비스에 적용되는
유료서비스 약관의 내용이 충돌하는 경우 개별 서비스에 적용되는 유료서비스 약관의
규정에 따릅니다.
제 11 조 (게시판 이용 상거래)
① 여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우
전자상거래 등에서의 소비자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을
준수하여야 합니다.
② 여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련
분쟁이 발생하는 경우, 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수
있는 장치를 마련합니다.
③ 회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의
신원정보를 확인하고, 여러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에
따라 소비자피해 분쟁조정기구, 공정거래위원회, 시·도지사 또는 시장·군수·구청장이
신원정보 제공을 요구하는 경우 이에 협조합니다.
제 12 조 (통합서비스 이용 방법 및 주의점)
① 여러분은 통합서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안
됩니다.
1. 이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 사람의 카카오계정 및
비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의
허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위
2. 회사의 서비스 정보를 이용하여 얻은 정보를 회사의 사전 승낙 없이 복제 또는
유통시키거나 상업적으로 이용하는 행위
3. 서비스 내에서 다운로드 또는 스트리밍을 통해 제공받은 음원을 사적 목적으로
이용하는 것 외에, 공공장소 및 영리를 목적으로 하는 영업장, 매장 등에서
재생하는 등의 방법으로 이용하는 행위
4. 타인의 명예를 손상시키거나 불이익을 주는 행위
5. 게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위
6. 회사 또는 제 3 자의 저작권 등 기타 권리를 침해하는 행위(국내외 제 3 자의
저작권 등을 침해하는 행위로서 회사가 IP 접속 차단 등 기술적인 조치를 통하여
타인에 대한 권리 침해 방지 조치를 취하였음에도 불구하고 이용자가 회사를
기망하는 수단과 방법 등을 통하여 서비스에 접속 하는 등 제 3 자의 저작권 등을
침해하는 행위를 포함합니다)
7. 서비스 내에 회사나 제 3 자 등에 대한 허위의 사실을 게시하는 행위
8. 공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게
유포하는 행위
9. 통합서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는
컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위
10. 통합서비스의 운영을 방해하거나 안정적 운영을 방해할 수 있는 정보 및
수신자의 명시적인 수신거부의사에 반하여 또는 수신자의 명시적인 동의 없이
광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위
11. 회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정,
배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와
소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해
또는 모방하거나 기타 변형하는 행위
12. 타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위
13. 다른 이용자의 개인정보를 수집, 저장, 공개하는 행위
14. 자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가하는 등
피해를 입힐 목적으로 허위의 정보를 유통시키는 행위
15. 재물을 걸고 도박하거나 사행행위를 하는 행위
16. 윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위
17. 수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는
영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는
행위
18. 관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램
포함)의 전송 또는 게시 행위
19. 회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의
명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위
20. 컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할
가능성이 있는 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을
포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위
21.스토킹(stalking), 허위 또는 악의적 신고 남용 등 다른 이용자를 괴롭히는 행위
22. 1 개월 이내 통합서비스 가입 및 유료서비스 구매 후 다시 해지하는 행위를
2 회 이상 반복하는 등 통합서비스를 부당하게 악용하는 행위
23. 기타 현행 법령, 본 약관 및 운영정책 등 회사가 제공하는 서비스 관련
세부지침을 위반하는 행위
② 여러분은 서비스의 이용 권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수
없으며, 담보로 제공할 수 없습니다.
③ 여러분의 자격 혹은 나이에 따라 아래 각 호처럼 통합서비스 이용의 일부가 제한될 수
있습니다.
1. 만19세 미만의 이용자는(단, 만 19세에 도달하는 해의 1월 1일을 맞이한 자는
제외, 이하 본 조에서 동일함) 정보통신망법 및 청소년보호법의 규정에 의하여
청소년유해매체물은 이용할 수 없습니다.
2. 청소년유해매체물을 이용하시기 위해서는 만 19 세 이상이어야 하며,
정보통신망법 및 청소년보호법의 규정에 의하여 실명인증을 통해 본인 및 연령
인증을 받으셔야 합니다. 인증을 받지 않으시면, 해당 서비스의 이용이
제한됩니다.
3. 만 19세 미만의 이용자의 서비스에 대하여 법정대리인의 요청 및 만19세 미만
이용자 본인의 동의가 있는 경우 개별 서비스의 전체 또는 일부의 이용이
일정기간 제한됩니다.
④ 회사는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우,
신고된 이용자의 음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다.
회사는 이용자간 분쟁 조정, 민원 처리를 위한 목적에 한하여, 제 3 자는 법령에 따라
권한이 부여된 경우에 한하여 이 정보를 열람할 수 있습니다. 회사는 부정이용 방지
및 관리의 목적에 따라 신고 접수시부터 3 년간 해당 정보를 3 년간 보관 후
파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수
있습니다.
⑤ 회사는 수사기관(경찰청 등)이 피싱 범죄에 이용중인 사실을 확인하여 법령 등에 따라
정당한 절차로 긴급차단을 요청하는 경우 여러분의 개별 서비스의 일부 또는 전부의
이용을 잠시 또는 계속하여 중단하는 이용 제한을 할 수 있습니다. 여러분이 이러한
이용 제한과 관련하여 이의가 있는 경우 이용정지를 요청한 수사기관에 이의제기를 할
수 있습니다. 수사기관에서 정당한 사유에 대한 소명이 확인된 경우 회사는 이용
제한을 해제할 수 있습니다.
⑥ 혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면,
회사는 여러분의 위반행위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시
삭제하거나 여러분의 계정·통합서비스 전체 또는 통합서비스를 구성하는 일부 개별
서비스의 이용을 잠시 또는 계속하여 중단하거나, 통합서비스 재가입 또는 일부 개별
서비스의 재이용에 제한을 둘 수도 있습니다. 또한 여러분이 통합서비스와 관련된
설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 통합서비스 제공에 악영향을
미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된
여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과
관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다.
⑦ 이용 제한은 위반 활동의 누적 정도에 따라 한시적 제한에서 영구적 제한으로 단계적
제한하는 것을 원칙으로 하지만, 음란한 내용의 게시와 유포 및 사행성 도박 홍보 등
관련 법령에서 금지하는 명백한 불법행위나 타인의 권리침해로서 긴급한 위험 또는
피해 차단이 요구되는 사안에 대해서는 위반 활동 횟수의 누적 정도와 관계 없이 즉시
영구적으로 이용이 제한될 수 있습니다.
⑧ 본 조에서 정한 사항 및 그 밖에 통합서비스의 이용에 관한 자세한 사항은 카카오
운영정책 등을 참고해 주시기 바랍니다.
제 13 조 (이용계약 해지)
① 여러분이 카카오계정 탈퇴를 하는 경우 통합서비스 이용계약도 자동으로 해지됩니다.
② 통합서비스 이용계약 해지를 원하는 경우 여러분은 언제든지 서비스 내 제공되는
메뉴를 이용하여 해지 신청을 할 수 있으며,회사는 법령이 정하는 바에 따라 신속히
처리하겠습니다.
③ 통합서비스 이용계약이 해지되면 관련 법령 및 카카오 개인정보 처리방침에 따라
여러분의 일정 정보를 보유하는 경우를 제외하고는 여러분의 정보나 여러분이 작성한
게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제 3 자에
의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제 3 자의 게시물에 댓글
등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글은 삭제되지 않으므로
반드시 해지 신청 전에 삭제하시기 바랍니다.
④ 전항에 따라 여러분이 삭제하지 않은 게시물은 다른 이용자의 정상적 서비스 이용을
위하여 필요한 범위 내에서 통합서비스 내에 삭제되지 않고 남아 있게 됩니다.
⑤ 유료서비스 이용계약의 해지는 여러분의 유료서비스 이용계약 해지 신청 및 회사의
승낙에 의해 성립하게 되고, 환불할 금액이 있는 경우 환불도 이루어 지게 됩니다.
다만 각 개별 서비스의 유료서비스에서 본 약관과 다른 계약해지 방법 및 효과를
규정하고 있는 경우 각 유료서비스 약관 및 관련 세부지침에서 정한 바에 따릅니다.
⑥ 통합서비스를 구성하는 일부 개별 서비스의 경우 일정기간 동안 해당 개별 서비스를
이용하지 않을 경우 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 또는 해당
개별 서비스 기능의 일부 또는 전부를 이용할 수 없도록 제한할 수 있습니다. 자세한
사항은 개별 서비스의 세부지침에서 확인하실 수 있습니다.
⑦ 통합서비스 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의
체결을 신청할 수 있습니다. 다만, 여러분이 관련 법령, 본 약관 및 세부지침을
준수하지 않아 서비스의 이용이 중단된 상태에서 이용계약을 해지한 후 다시 이용계약
체결을 신청하는 경우에는 통합서비스 가입에 일정기간 시간적 제한이 있을 수
있습니다. 또한 통합서비스를 구성하는 일부 개별 서비스의 경우 다시 통합서비스
이용계약을 체결한 후에도 해당 개별 서비스를 바로 이용하는 것에는 제 6 조
제 3 항에서 정한 바와 같이 일정한 시간적 제한 등이 따를 수 있습니다.
제 14 조 (개인정보의 보호)
여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의
개인정보는 통합서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만
이용됩니다. 관련 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의
개인정보를 제 3 자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의
개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 카카오
개인정보처리방침 등을 참고해 주시기 바랍니다.
제 4 장 기타
제 15 조 (손해배상 등)
① 회사는 관련 법령상 허용되는 한도 내에서 통합서비스와 관련하여 본 약관에 명시되지
않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는
CP(Contents Provider)가 제공하거나 여러분이 작성하는 등의 방법으로 통합서비스에
게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의
과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다.
② 회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에
따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와
같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도
내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한
책임을 부담하지 않습니다.
1. 천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해
2. 여러분의 귀책사유로 통합서비스 이용에 장애가 발생한 경우
3. 통합서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해
4. 제 3 자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는
손해
5. 제 3 자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써
발생하는 손해
6. 제 3 자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해
7. 전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제 3 자가
서비스를 이용하는 과정에서 발생된 손해
8. 기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해
③ 회사는 회사의 고의 또는 과실이 없는 한 여러분이 통합서비스를 이용하여 기대하는
수익을 상실한 것에 대하여 책임을 지지 않으며 그 밖에 통합서비스를 통하여 얻은
자료로 인한 손해 등에 대하여도 책임을 지지 않습니다.
④ 회사는 회사의 과실이 없는 한 여러분 상호간 또는 여러분과 제 3 자 상호간에
통합서비스를 매개로 발생한 분쟁에 대해서는 개입할 의무가 없으며 이로 인한 손해를
배상할 책임도 없습니다.
제 16 조 (청소년보호)
통합서비스는 기본적으로 모든 연령대가 자유롭게 이용할 수 있는 공간으로서 유해 정보로부터
청소년을 보호하고 청소년의 안전한 인터넷 사용을 돕기 위해 정보통신망법에서 정한
청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은 통합서비스를 구성하는 개별 서비스
초기 화면 등에서 확인할 수 있습니다.
제 17 조 (통지 및 공지)
회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 카카오 고객센터에
방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스
공지사항 란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의
경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내
전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이
등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려
드리겠습니다.
제 18 조 (분쟁의 해결)
본 약관 또는 통합서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 통합서비스 이용과
관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다.
그럼에도 불구하고 해결되지 않으면 민사소송법의 관할법원에 소를 제기할 수 있습니다.
• 공고일자 : 2026 년 5 월 13 일
• 시행일자 : 2026 년 5 월 29 일""" }   # 약관 4종 원문


# ── 정규식 ────────────────────────────────────────────────
ARTICLE_RE = re.compile(
    r"^[ \t]*제\s*(?P<num>\d+)\s*조"
    r"(?:\s*의\s*(?P<sub>\d+))?"
    r"(?:"
    r"[ \t]*[(（]\s*(?P<title_p>[^)）\n]*?)\s*[)）]"      # 제7조 (계정의 해지)
    r"|"
    r"[ \t]+(?P<title_b>[^\n]{1,40})[ \t]*(?=\n|$)"      # 제1조 목적
    r")?",
    re.MULTILINE,
)
SUPPLEMENT_RE = re.compile(r"^\s*부\s*칙", re.MULTILINE)
CHAPTER_LINE  = re.compile(r"^[ \t]*제\s*(\d+)\s*장[ \t]*([^\n]*)$", re.MULTILINE)

# =====================================================================================
# 튜닝 파라미터 — 실험할 때는 우선 이 구역의 값만 변경하세요.
# 한 번에 하나만 바꾸고 결과 파일과 점수를 version.md에 기록해야 원인을 비교할 수 있습니다.
# 아래 기본값은 실험 28의 동작 조건입니다.
# =====================================================================================

# 1) 모델 선택·메모리
EMBED_MODEL = "BAAI/bge-m3"       # 검색 임베딩 모델
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"  # USE_RERANKER=True일 때만 로드
GEN_MODEL = REQUIRED_GENERATION_MODEL_FAMILY  # 대회가 허용한 Qwen 계열 생성 모델
USE_RERANKER = False                 # True: 검색 정밀도 가능성↑, VRAM·지연↑
LOAD_IN_4BIT = True                  # T4에서 7B 모델을 올리기 위한 4-bit 로딩
QUANTIZATION_TYPE = "nf4"          # 4-bit 양자화 방식
USE_DOUBLE_QUANT = True              # 추가 압축으로 VRAM 절약
GEN_COMPUTE_DTYPE = torch.float16    # T4 계산 자료형
DEVICE_MAP = "cuda"               # 모델 배치 장치

# 2) 코퍼스·검색
CHUNK_MAX_CHARS = 1200               # 긴 조항 조각의 최대 문자 수
BM25_K1 = 1.5                        # 단어 반복 빈도의 포화 속도
BM25_B = 0.75                        # 문서 길이 정규화 강도
RETRIEVER_POOL = 20                  # BM25·임베딩의 1차 후보 수
RETRIEVER_TOP_K = 4                  # 반환할 최대 조항 수
RETRIEVER_SCORE_RATIO = 0.45         # 1위 점수 대비 후보 유지 비율
RRF_K = 60                           # BM25·임베딩 순위 융합 완화 상수
EMBED_BATCH_SIZE = 16                # 문서 임베딩 배치 크기
RERANK_BATCH_SIZE = 8                # 리랭커 배치 크기
RERANK_MAX_LENGTH = 512              # 리랭커 최대 입력 토큰

# 3) 생성에 강조할 원문 구절
FOCUS_MAX_SENTENCES = 3              # 일반 질문에서 생성 모델에 강조하는 문장 수
FOCUS_NUMBER_BONUS = 0.5             # 질문과 같은 숫자가 있는 문장 보너스
FOCUS_HIT_RANK_PENALTY = 0.05        # 검색 순위가 낮은 조항의 감점
LIST_MIN_COUNT = 2                   # 질문이 요구한 번호 목록의 최소 개수
LIST_MAX_COUNT = 20                  # 비정상적으로 큰 목록 요청 차단 상한

# 4) 실험 28 누락 원문 1문장 후처리
MISSING_EVIDENCE_SOURCE_HITS = 1     # 1위 검색 조항만 검사
MISSING_EVIDENCE_CANDIDATES = 5      # 내부 후보는 넓게 보고 최종 1문장만 선택
MISSING_EVIDENCE_ANSWER_COVERAGE = 0.60
MISSING_EVIDENCE_DIRECT_QUESTION_COVERAGE = 0.45
MISSING_EVIDENCE_STRONG_QUESTION_COVERAGE = 0.30
MISSING_EVIDENCE_STRONG_ANSWER_COVERAGE = 0.40
MISSING_SCORE_QUESTION_WEIGHT = 0.25  # 질문 관련성 비중
MISSING_SCORE_NOVELTY_WEIGHT = 0.35   # 기존 답에 없는 정보 비중
MISSING_SCORE_ROLE_WEIGHT = 0.40      # 질문 유형에 필요한 법적 역할 비중
MISSING_SCORE_LENGTH_PENALTY = 0.15   # 긴 원문 전체 추가 억제
MISSING_SCORE_NEGATION_PENALTY = 0.25 # 질문에 없던 부정·불가 문장 억제
MISSING_SCORE_MINIMUM = 0.48          # 이 점수 미만이면 아무 문장도 추가하지 않음
MISSING_CLAUSE_MIN_CHARS = 20         # 너무 짧은 절 조각 사용 방지

# 5) 답변 생성
GEN_MAX_NEW_TOKENS = 350             # 답변 최대 생성 토큰
GEN_DO_SAMPLE = False                # False: 재현성 높은 결정적 생성


# ── 파싱 ──────────────────────────────────────────────────
def _clean(text: str) -> str:
    """약관 원문의 줄바꿈과 공백을 정규화해 파싱 가능한 문자열로 반환합니다."""
    text = text.replace("\r\n", "\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def parse_articles(raw: dict[str, str], drop_supplement: bool = True) -> list[dict[str, object]]:
    """문서별 원문을 조 단위 딕셔너리 목록으로 변환합니다.

    Args:
        raw: 공식 문서명을 키, 약관 전문을 값으로 갖는 딕셔너리.
        drop_supplement: True이면 시행일·부칙 이후 내용을 검색 코퍼스에서 제외합니다.
    Returns:
        doc, article, sub, title, text를 담은 조항 목록.
    """
    chunks = []
    for doc, body in raw.items():
        if doc not in OFFICIAL_DOCUMENT_NAMES:
            raise ValueError(f"문서명이 공식 명칭과 다릅니다: {doc!r}")

        body = _clean(body)
        m = SUPPLEMENT_RE.search(body)
        if m and drop_supplement:
            body = body[:m.start()]

        matches = list(ARTICLE_RE.finditer(body))
        if not matches:
            raise ValueError(f"{doc}: 조 헤더를 찾지 못했습니다.")

        for i, mt in enumerate(matches):
            end = matches[i + 1].start() if i + 1 < len(matches) else len(body)
            text = body[mt.end():end].strip()
            if not text:
                continue
            chunks.append({
                "doc": doc,
                "article": int(mt.group("num")),
                "sub": int(mt.group("sub")) if mt.group("sub") else None,
                "title": (mt.group("title_p") or mt.group("title_b") or "").strip(),
                "text": text,
            })
    return chunks

def extract_chapter(chunks: list[dict[str, object]]) -> list[dict[str, object]]:
    """조 본문 끝의 장 헤더를 분리하고 다음 조부터 chapter 메타데이터로 붙입니다."""
    out, current = [], {}
    for c in chunks:
        m = CHAPTER_LINE.search(c["text"])
        text = re.sub(r"\n{3,}", "\n\n", CHAPTER_LINE.sub("", c["text"])).strip()
        if len(text) >= 20:
            out.append({**c, "text": text, "chapter": current.get(c["doc"], "")})
        if m:
            current[c["doc"]] = m.group(2).strip()
    return out

def split_long(
    chunks: list[dict[str, object]],
    max_chars: int = 1200,
) -> list[dict[str, object]]:
    """긴 조항을 번호·문장 경계에서 나누고 조항 메타데이터를 유지합니다."""
    PART = r"(?=[①-⑳]|(?:^|\n)\s*\d{1,2}\.\s|(?:^|\n)\s*[(（]\d{1,2}[)）])"

    def split_oversized(text: str) -> list[str]:
        """번호 경계로 나뉘지 않는 긴 문단도 문장·줄 경계에서 상한 이하로 자릅니다."""
        parts = []
        remaining = text.strip()
        while len(remaining) > max_chars:
            window = remaining[:max_chars + 1]
            candidates = [
                window.rfind("\n"),
                window.rfind("다.") + 2,
                window.rfind("요.") + 2,
                window.rfind(". ") + 1,
            ]
            cut = max(candidates)
            if cut < max_chars // 2:
                cut = max_chars
            parts.append(remaining[:cut].strip())
            remaining = remaining[cut:].strip()
        if remaining:
            parts.append(remaining)
        return parts

    out = []
    for c in chunks:
        if len(c["text"]) <= max_chars:
            out.append(c); continue
        buf = ""
        for p in [x for x in re.split(PART, c["text"]) if x and x.strip()]:
            if len(p) > max_chars:
                if buf.strip():
                    out.append({**c, "text": buf.strip()})
                    buf = ""
                out.extend({**c, "text": part} for part in split_oversized(p))
                continue
            if buf and len(buf) + len(p) > max_chars:
                out.append({**c, "text": buf.strip()}); buf = p
            else:
                buf += p
        if buf.strip():
            out.append({**c, "text": buf.strip()})
    return out

# print(f"조각 {len(CHUNKS)}개")

# --------------------------------------------------
# RAG
# =====================================================================================
#  1번 셀 — 검색(R) + 생성(G) 구현
#  전제: CHUNKS 가 파싱 단계에서 아래 형태로 이미 만들어져 있음
#    [{"doc": "카카오계정 약관", "article": 7, "title": "계정의 해지", "text": "① 회원은..."}, ...]
# =====================================================================================

# -------------------------------------------------------------------------------------
# 1. 한국어 토크나이저 — BM25용 TOKENIZE
# -------------------------------------------------------------------------------------
# 공백 분리만 하면 "해지를 / 해지가 / 해지는"이 전부 다른 토큰이 되어 매칭률이 급락한다.
# 형태소 분석으로 어간만 남긴다. 설치 실패 시 정규식 폴백.

try:
    from kiwipiepy import Kiwi
    _kiwi = Kiwi()
    _KEEP = ("NNG", "NNP", "NNB", "SL", "SH", "SN", "VV", "VA", "XR", "MAG")

    def tokenize(text: str) -> list[str]:
        """Kiwi 형태소 분석으로 검색에 사용할 내용어 토큰만 반환합니다."""
        return [t.form for t in _kiwi.tokenize(text) if t.tag in _KEEP]

except Exception as e:
    print(f"[warn] kiwipiepy 사용 불가 → 정규식 폴백 ({e})")
    _JOSA = re.compile(r"(을|를|이|가|은|는|의|에|에서|으로|로|와|과|도|만|까지|부터|에게|께|한테)$")

    def tokenize(text: str) -> list[str]:
        """Kiwi를 사용할 수 없을 때 정규식과 조사 제거로 토큰을 반환합니다."""
        toks = re.findall(r"[가-힣]+|[a-zA-Z]+|\d+", text)
        return [_JOSA.sub("", t) for t in toks if len(t) > 1]

# -------------------------------------------------------------------------------------
# 2. 검색기 RETRIEVER
# -------------------------------------------------------------------------------------

class Retriever:
    """BM25(어휘) + 임베딩(의미)을 RRF로 융합. 선택적으로 리랭커 적용."""

    def __init__(
        self,
        chunks: list[dict[str, object]],
        use_reranker: bool = USE_RERANKER,
    ) -> None:
        """조항 코퍼스의 BM25·Dense 인덱스를 만들고 선택적으로 리랭커를 로드합니다."""
        self.chunks = chunks

        # 검색용 텍스트: 문서명 + 조 제목을 본문 앞에 붙여 맥락을 준다.
        # 조 제목이 사실상 그 조의 요약이라 매칭 기여도가 크다.
        self.search_texts = [
            f"{c['doc']} 제{c['article']}조 {c.get('title', '')}\n{c['text']}"
            for c in chunks
        ]

        # --- 어휘 검색 ---
        self.bm25 = BM25Okapi([tokenize(t) for t in self.search_texts], k1=BM25_K1, b=BM25_B)

        # --- 의미 검색 ---
        self.embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE_MAP)
        self.doc_vecs = self.embedder.encode(
            self.search_texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,      # 정규화해두면 내적 = 코사인 유사도
            show_progress_bar=True,
        )

        # --- 리랭커 (선택) ---
        self.reranker = None
        if use_reranker:
            try:
                from sentence_transformers import CrossEncoder
                self.reranker = CrossEncoder(RERANK_MODEL, device=DEVICE_MAP, max_length=RERANK_MAX_LENGTH)
            except Exception as e:
                print(f"[warn] 리랭커 로드 실패 → 비활성화 ({e})")

    # ---------------------------------------------------------------------------------
    def _rrf(self, rank_lists: list[list[int]], k: int = RRF_K) -> list[int]:
        """Reciprocal Rank Fusion — 점수 스케일이 다른 랭킹들을 순위만으로 합친다.
        정규화·가중치 튜닝이 필요 없어 실패 위험이 낮다."""
        fused = {}
        for ranks in rank_lists:
            for pos, idx in enumerate(ranks):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + pos + 1)
        return sorted(fused, key=fused.get, reverse=True)

    # ---------------------------------------------------------------------------------
    def search(
        self,
        question: str,
        pool: int = RETRIEVER_POOL,
        top_k: int = RETRIEVER_TOP_K,
        ratio: float = RETRIEVER_SCORE_RATIO,
    ) -> list[dict[str, object]]:
        """질문과 관련된 조항을 하이브리드 검색해 점수순으로 반환합니다.

        Args:
            question: 사용자가 입력한 한국어 약관 질문.
            pool: BM25·Dense 검색과 선택적 리랭킹에 사용할 1차 후보 수.
            top_k: 최종 반환할 최대 조항 수.
            ratio: 1위 점수 대비 후보 유지 비율.
        Returns:
            원본 조항 메타데이터와 score를 포함한 검색 결과 목록.
        """

        # 1차: BM25 랭킹
        bm_scores = self.bm25.get_scores(tokenize(question))
        bm_rank = np.argsort(bm_scores)[::-1][:pool].tolist()

        # 1차: 임베딩 랭킹
        q_vec = self.embedder.encode(
            question, convert_to_tensor=True, normalize_embeddings=True
        )
        sims = (self.doc_vecs @ q_vec).cpu().numpy()
        emb_rank = np.argsort(sims)[::-1][:pool].tolist()

        # 융합
        cand = self._rrf([bm_rank, emb_rank])[:pool]

        # 2차: 리랭킹 (질문-조항 쌍을 함께 읽어 정밀 재정렬)
        if self.reranker is not None and cand:
            pairs = [(question, self.search_texts[i][:1500]) for i in cand]
            scores = self.reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE)
            order = np.argsort(scores)[::-1]
            cand = [cand[i] for i in order]
            scores = [float(scores[i]) for i in order]
        else:
            scores = [1.0 / (r + 1) for r in range(len(cand))]

        # 조 단위로 되접기 — 같은 조의 여러 항이 올라오면 최고 점수만 남긴다.
        # retrieved는 조번호 단위로 나가야 하므로 여기서 중복을 제거한다.
        seen, hits = {}, []
        for idx, sc in zip(cand, scores):
            c = self.chunks[idx]
            key = (c["doc"], c["article"])
            if key in seen:
                seen[key]["text"] += "\n" + c["text"]     # 같은 조의 다른 항은 본문에 이어붙임
                continue
            item = {**c, "score": sc}
            seen[key] = item
            hits.append(item)
            if len(hits) >= top_k:
                break

        # 확신 없는 조항 잘라내기 — 무조건 4개를 내면 정밀도가 깎인다.
        if hits:
            cut = hits[0]["score"] * ratio
            hits = [h for h in hits if h["score"] >= cut] or hits[:1]

        return hits

# ------------------------------------
# GENERATOR
# -------------------------------------------------------------------------------------
# 3. 생성기
# -------------------------------------------------------------------------------------

SYSTEM = (
    "당신은 카카오 약관 안내 담당자입니다. 반드시 한국어로만 답하세요.\n"
    "규칙:\n"
    "1. 제공된 약관 조항에 적힌 내용만으로 답하세요. 조항에 없는 내용은 어떤 경우에도 덧붙이지 마세요.\n"
    "2. 답하기 전에 조항에서 근거 문장을 찾고, 그 문장의 결론대로 답하세요. 조항의 결론이 상식과 반대여도 조항을 따르세요.\n"
    "3. 질문이 조항과 다른 전제를 담고 있으면, 그 전제가 틀렸음을 먼저 밝히고 조항의 실제 내용을 설명하세요.\n"
    "4. 조항에 수치, 기간, 법조문 번호, 열거 항목이 있으면 하나도 빠뜨리지 말고 그대로 포함하세요.\n"
    "5. 질문에 답하는 데 불필요한 부연이나 일반론은 덧붙이지 마세요.\n"
    "6. 조항 번호나 '제N조' 같은 표현은 답변에 쓰지 마세요."
)

def select_focus_sentences(
    question: str,
    hits: list[dict[str, object]],
    max_sentences: int = FOCUS_MAX_SENTENCES,
) -> list[str]:
    """검색 조항에서 질문과 가까운 원문 문장을 골라 생성 모델에 강조합니다.

    질문 토큰의 희소성, 같은 숫자, 검색 순위를 점수화하며 명시적 목록 질문은
    요구 개수만큼의 번호 항목을 함께 보존합니다.
    """
    q_tokens = set(tokenize(question))
    if not q_tokens:
        return []

    candidates = []
    seen = set()
    for hit_rank, hit in enumerate(hits):
        # PDF 줄바꿈은 문장 중간에서도 생기므로 번호·글머리표 경계만 보존합니다.
        text = re.sub(r"\n(?!\s*(?:[①-⑳]|\d{1,2}\.|•))", " ", hit["text"])
        sentences = re.split(r"(?<=[.!?])\s+|\n+", text)
        for order, sentence in enumerate(sentences):
            sentence = re.sub(r"\s+", " ", sentence).strip()
            normalized = re.sub(r"\s+", "", sentence)
            if len(sentence) < 12 or normalized in seen:
                continue
            tokens = set(tokenize(sentence))
            overlap = q_tokens & tokens
            if not overlap:
                continue
            seen.add(normalized)
            candidates.append({
                "sentence": sentence,
                "tokens": tokens,
                "overlap": overlap,
                "hit_rank": hit_rank,
                "order": order,
            })

    if not candidates:
        return []

    # 여러 문장에 흔한 단어보다 특정 문장에만 있는 질문 단어를 더 높게 평가합니다.
    doc_freq = {t: sum(t in c["tokens"] for c in candidates) for t in q_tokens}
    for c in candidates:
        lexical = sum(np.log((len(candidates) + 1) / (doc_freq[t] + 1)) + 1.0 for t in c["overlap"])
        number_bonus = FOCUS_NUMBER_BONUS * len(set(re.findall(r"\d+", question)) & set(re.findall(r"\d+", c["sentence"])))
        c["score"] = lexical / max(1.0, np.sqrt(len(c["tokens"]))) + number_bonus - FOCUS_HIT_RANK_PENALTY * c["hit_rank"]

    ranked = sorted(candidates, key=lambda c: (-c["score"], c["hit_rank"], c["order"]))

    selected = ranked[:max_sentences]

    # 질문이 항목 개수를 명시하고 원문에 같은 번호 목록이 있으면 목록을 전부 강조합니다.
    count_match = re.search(r"(\d{1,2})\s*(?:가지|개|항목|종류)", question)
    if count_match:
        requested_count = int(count_match.group(1))
        if LIST_MIN_COUNT <= requested_count <= LIST_MAX_COUNT:
            numbered = {}
            for c in candidates:
                item_match = re.match(r"^\s*(\d{1,2})[.)]\s*", c["sentence"])
                if item_match:
                    numbered.setdefault(int(item_match.group(1)), c)
            required = [numbered[i] for i in range(1, requested_count + 1) if i in numbered]
            if len(required) == requested_count:
                required_ids = {id(c) for c in required}
                selected = required

    return [c["sentence"] for c in selected]


def _missing_role_terms(question: str) -> set[str]:
    """질문의 법적 역할에 대응하는 완결성 검사 표현 집합을 반환합니다."""
    groups = (
        (r"중단|장애|지연|복구|공지", ("복구", "노력", "공지", "통지", "지연")),
        (r"효력|존속|탈퇴|사용을 중단", ("효력", "존속", "영구적", "전 세계적", "간주")),
        (r"충돌|우선|세부지침|규정되지", ("충돌", "우선", "세부지침", "규정되지", "따릅니다")),
        (r"포함|제공하는 서비스|누가 제공|가입", ("포함되지", "제외", "제3자", "계열사")),
        (r"제출|서류|첨부|절차", ("제출", "첨부", "서면", "증명", "절차")),
        (r"기간|개월|시간|며칠", ("기간", "개월", "시간", "즉시", "지체없이")),
    )
    terms = set()
    for pattern, values in groups:
        if re.search(pattern, question):
            terms.update(values)
    return terms


def _score_missing_evidence(
    question: str,
    answer: str,
    sentence: str,
) -> tuple[float, float, float]:
    """누락 후보 문장의 종합점수·질문 포함률·기존 답변 포함률을 반환합니다.

    질문 관련성보다 기존 답에 없는 법적 역할을 더 크게 평가하고, 긴 문장과
    질문에 없던 부정 표현에는 감점을 적용합니다.
    """
    q_tokens = set(tokenize(question))
    a_tokens = set(tokenize(answer))
    s_tokens = set(tokenize(sentence))
    if not q_tokens or not s_tokens:
        return -1.0, 0.0, 1.0

    question_coverage = len(s_tokens & q_tokens) / max(1, len(q_tokens))
    answer_coverage = len(s_tokens & a_tokens) / len(s_tokens)
    question_score = min(1.0, question_coverage / max(0.01, MISSING_EVIDENCE_DIRECT_QUESTION_COVERAGE))
    novelty_score = 1.0 - answer_coverage

    role_terms = _missing_role_terms(question)
    answer_roles = {term for term in role_terms if term in answer}
    new_roles = {term for term in role_terms if term in sentence and term not in answer_roles}
    role_score = len(new_roles) / max(1, min(2, len(role_terms))) if role_terms else 0.0
    role_score = min(1.0, role_score)

    length_penalty = max(0.0, len(sentence) - 180) / 300.0
    candidate_has_negation = bool(re.search(r"없|않|못|불가|금지", sentence))
    question_has_negation = bool(re.search(r"없|않|못|불가|금지", question))
    answer_has_negation = bool(re.search(r"없|않|못|불가|금지", answer))
    negation_penalty = MISSING_SCORE_NEGATION_PENALTY if candidate_has_negation and not (question_has_negation or answer_has_negation) else 0.0

    score = (
        MISSING_SCORE_QUESTION_WEIGHT * question_score
        + MISSING_SCORE_NOVELTY_WEIGHT * novelty_score
        + MISSING_SCORE_ROLE_WEIGHT * role_score
        - MISSING_SCORE_LENGTH_PENALTY * length_penalty
        - negation_penalty
    )
    return score, question_coverage, answer_coverage


def _extract_relevant_clause(question: str, answer: str, sentence: str) -> str:
    """긴 문장은 명확한 담화 접속 경계가 있을 때만 가장 관련된 절로 줄입니다."""
    cleaned = re.sub(r"^[•①-⑳]\s*", "", sentence).strip()
    parts = [
        part.strip(" ,")
        for part in re.split(r"\s+(?=(?:다만|또한|그리고|이 경우|따라서)[,\s])", cleaned)
        if len(part.strip(" ,")) >= MISSING_CLAUSE_MIN_CHARS
    ]
    if len(parts) <= 1:
        return cleaned
    return max(parts, key=lambda part: _score_missing_evidence(question, answer, part)[0])


def remove_embedded_duplicate_sentences(answer: str) -> str:
    """다른 문장 안에 내용이 완전히 포함된 긴 중복 문장만 제거합니다."""
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+|\n+", answer)
        if sentence.strip()
    ]
    normalized = [re.sub(r"[\s.!?]+", "", sentence) for sentence in sentences]
    keep = [True] * len(sentences)

    for i, first in enumerate(normalized):
        if not keep[i]:
            continue
        for j in range(i + 1, len(normalized)):
            if not keep[j]:
                continue
            second = normalized[j]
            if first == second:
                keep[j] = False
            elif len(first) >= 30 and first in second:
                keep[i] = False
                break
            elif len(second) >= 30 and second in first:
                keep[j] = False

    return " ".join(sentence for sentence, retained in zip(sentences, keep) if retained)


def format_natural_answer(answer: str) -> str:
    """내용은 바꾸지 않고 최종 출력을 한 문단의 자연스러운 답변으로 정리합니다."""
    # 생성 답변과 후처리 원문 사이에 포함 관계가 있으면 더 완전한 문장 하나만 남깁니다.
    answer = remove_embedded_duplicate_sentences(answer)
    # 여러 줄과 빈 줄을 한 칸으로 합쳐 JSON에서도 실제 답변처럼 보이게 합니다.
    answer = re.sub(r"\s*\n+\s*", " ", answer).strip()
    # 약관 정의어를 둘러싼 장식용 따옴표만 제거합니다.
    answer = answer.translate(str.maketrans("", "", "'‘’“”\""))
    # 목록 내용은 유지하고 문장 앞의 1. 2. 같은 출력 표지만 제거합니다.
    answer = re.sub(r"(?:(?<=^)|(?<=\s))\d{1,2}\.\s+", "", answer)
    return re.sub(r"[ \t]+", " ", answer).strip()


def append_missing_evidence(
    question: str,
    answer: str,
    hits: list[dict[str, object]],
) -> str:
    """1위 근거의 핵심 문장이 빠진 경우 가장 안전한 원문 한 문장만 보완합니다.

    질문 관련성·답변 대비 새 정보·법적 역할을 함께 평가하며 임계값을 넘지
    못하면 입력 답변을 변경하지 않습니다.
    """
    if not hits:
        return answer

    q_tokens = set(tokenize(question))
    a_tokens = set(tokenize(answer))
    if not q_tokens:
        return answer

    # 다른 후보 조항과 낮은 순위 문장은 제외해 F1 정밀도 하락을 제한합니다.
    focus = select_focus_sentences(
        question,
        hits[:MISSING_EVIDENCE_SOURCE_HITS],
        max_sentences=MISSING_EVIDENCE_CANDIDATES,
    )[:MISSING_EVIDENCE_CANDIDATES]
    scored = []
    role_terms = _missing_role_terms(question)
    for sentence in focus:
        if role_terms and not any(term in sentence for term in role_terms):
            continue
        score, question_coverage, answer_coverage = _score_missing_evidence(question, answer, sentence)
        is_direct = question_coverage >= MISSING_EVIDENCE_DIRECT_QUESTION_COVERAGE
        is_strong_omission = (
            question_coverage >= MISSING_EVIDENCE_STRONG_QUESTION_COVERAGE
            and answer_coverage < MISSING_EVIDENCE_STRONG_ANSWER_COVERAGE
        )
        if answer_coverage < MISSING_EVIDENCE_ANSWER_COVERAGE and (is_direct or is_strong_omission):
            scored.append((score, sentence))

    if not scored:
        return answer
    best_score, best_sentence = max(scored, key=lambda item: item[0])
    if best_score < MISSING_SCORE_MINIMUM:
        return answer
    evidence = _extract_relevant_clause(question, answer, best_sentence)
    if not evidence:
        return answer
    return f"{answer.rstrip()}\n\n{evidence}"


def append_required_legal_conclusions(
    question: str,
    answer: str,
    hits: list[dict[str, object]],
) -> str:
    """복합 질문에 빠진 동의 효력·서비스 범위 결론을 1위 원문에서 보완합니다.

    원문에 목표 표현이 실제로 있고 답변에는 없을 때만 결론절을 추가합니다.
    P07·P10에서 관찰된 누락을 문항 ID 없이 질문 유형으로 일반화한 안전장치입니다.
    """
    if not hits:
        return answer

    evidence = re.sub(r"\s+", " ", hits[0]["text"]).strip()
    additions = []

    # 동의의 효력을 직접 묻는데 간주 결론이 빠진 경우 원문 결론절만 보존합니다.
    asks_consent_effect = "동의" in question and "효력" in question
    has_consent_effect = "본인의 동의" in answer or "것으로 봅니다" in answer
    effect_match = re.search(r"본인의\s+동의가\s+있는\s+것으로\s+봅니다", evidence)
    if asks_consent_effect and not has_consent_effect and effect_match:
        additions.append("이 경우 " + re.sub(r"\s+", " ", effect_match.group(0)) + ".")

    # 서비스의 포함 범위를 묻는 경우 같은 1위 정의 조항의 제3자 비가입 효과도 보존합니다.
    asks_service_scope = bool(re.search(r"포함.*서비스|서비스.*포함|누가\s*제공", question))
    has_non_membership = bool(re.search(r"가입되지는\s*않|가입되지\s*않", answer))
    scope_match = re.search(
        r"(회사가\s+아닌\s+계열사를\s+포함한\s+제\s*3\s*자가\s+제공하는\s+서비스).*?가입되지는\s+않으며",
        evidence,
    )
    if asks_service_scope and not has_non_membership and scope_match:
        subject = re.sub(r"\s+", " ", scope_match.group(1)).replace("제 3 자", "제3자")
        additions.append(f"여러분은 {subject}에 가입되지는 않습니다.")

    if not additions:
        return answer
    return answer.rstrip() + "\n\n" + " ".join(additions)


class Generator:
    """검색 근거로 Qwen 답변을 생성하고 누락 보완·출력 정리를 수행합니다."""

    def __init__(self) -> None:
        """토크나이저와 4-bit Qwen 생성 모델을 GPU에 로드합니다."""
        self.tok = AutoTokenizer.from_pretrained(GEN_MODEL)
        self.model = AutoModelForCausalLM.from_pretrained(
            GEN_MODEL,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=LOAD_IN_4BIT,
                bnb_4bit_compute_dtype=GEN_COMPUTE_DTYPE,
                bnb_4bit_quant_type=QUANTIZATION_TYPE,
                bnb_4bit_use_double_quant=USE_DOUBLE_QUANT,
            ),
            device_map=DEVICE_MAP,
        ).eval()

    def generate(
        self,
        question: str,
        hits: list[dict[str, object]],
        max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    ) -> str:
        """검색 조항으로 프롬프트를 구성해 최종 한국어 답변 문자열을 반환합니다."""
        if not hits:
            return "해당 질문에 대한 근거 조항을 약관에서 찾지 못했습니다."

        # 전체 조항과 별도의 짧은 focus 블록을 전달합니다.
        context = "\n\n".join(
            f"[조항 {i}] {h.get('title', '')}\n{h['text']}" for i, h in enumerate(hits, 1)
        )
        focus = select_focus_sentences(question, hits)
        focus_block = "\n".join(f"- {sentence}" for sentence in focus)
        user = (f"[약관 조항]\n{context}\n\n"
                f"[질문]\n{question}\n\n"
                f"[질문과 직접 관련된 원문 구절]\n{focus_block}\n\n"
                "위 조항에서 근거 문장을 찾은 뒤, 그 문장의 결론대로 한국어로만 답하세요. "
                "관련 원문 구절을 먼저 확인하되 전체 조항과 함께 판단하고, 질문에 직접 답하는 핵심 표현은 생략하지 마세요. "
                "질문의 전제가 조항과 다르면 바로잡고, 근거에 없는 내용은 추가하지 마세요.")

        prompt = self.tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = self.tok(prompt, return_tensors="pt").to("cuda")

        with torch.inference_mode():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=GEN_DO_SAMPLE,
                pad_token_id=self.tok.eos_token_id,
            )
        answer = self.tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
        answer = append_missing_evidence(question, answer, hits)
        answer = append_required_legal_conclusions(question, answer, hits)
        return format_natural_answer(answer)


# ── 초기화 ─────────────────────────────
CHUNKS = split_long(extract_chapter(parse_articles(RAW)), max_chars=CHUNK_MAX_CHARS)
RETRIEVER = Retriever(CHUNKS)
GENERATOR = Generator()

def answer_question(question: str) -> dict[str, object]:
    """질문을 검색·생성하고 대회 계약 형식의 answer와 retrieved를 반환합니다."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")

    hits = RETRIEVER.search(question.strip())

    return {
        "answer": GENERATOR.generate(question.strip(), hits),
        # 조번호는 검색 결과에서 그대로 옮겨 담는다. 모델 출력에서 파싱하지 않는다.
        "retrieved": [[h["doc"], h["article"]] for h in hits[:4]],
    }



# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")